# 01. Limpieza e integración de datos

**Notebook anterior:** `00_inicializacion_datos_movielens.ipynb`

### Introducción a la etapa de limpieza e integración

Después de completar la adquisición, validación y comprensión inicial del conjunto de datos MovieLens 32M, se inicia la primera fase de preparación de datos orientada al análisis.

Durante la etapa anterior se verificó la integridad de los archivos descargados, se inspeccionó la estructura general de los datos y se realizó una caracterización preliminar de las tablas principales. Sin embargo, antes de comenzar cualquier análisis estadístico o exploratorio avanzado, resulta necesario evaluar la calidad de los datos, comprender las relaciones existentes entre las entidades del modelo y definir una estrategia de integración que preserve la semántica original de la información.

Esta fase constituye el puente entre la adquisición de datos y el análisis exploratorio formal, asegurando que las decisiones posteriores se realicen sobre estructuras limpias, documentadas y reproducibles.

### Objetivo de la preparación del entorno

Antes de iniciar las actividades de limpieza e integración, es necesario configurar nuevamente el entorno de trabajo.

Aunque los datos ya fueron cargados e inspeccionados en el notebook anterior, este proceso se diseña bajo criterios de reproducibilidad. Por esta razón, el notebook vuelve a establecer todas las dependencias, rutas y configuraciones necesarias para funcionar de manera autónoma.

De esta forma:

- No depende de variables previamente almacenadas en memoria.
- Puede ejecutarse de forma independiente por cualquier integrante del equipo.
- Garantiza resultados consistentes entre distintas sesiones de trabajo.
- Facilita la replicación completa del proyecto por terceros.

### Importación de bibliotecas

En esta etapa se incorporan bibliotecas orientadas tanto a la manipulación de datos como a la gestión de archivos y diagnósticos de calidad.

#### Manipulación y análisis de datos

- **Pandas (`pandas`)**: será la biblioteca principal para limpieza, transformación e integración de tablas.
- **NumPy (`numpy`)**: proporcionará soporte para operaciones numéricas y manejo eficiente de estructuras matriciales.

#### Gestión de directorios y archivos

- **Pathlib (`Path`)**: permite construir rutas independientes del sistema operativo.
- **Shutil (`shutil`)**: incorpora utilidades para copias y operaciones sobre archivos y directorios.
- **zipfile (`Zipfile`)**: permite acceder a archivos comprimidos de manera rapida y sencilla.

#### Diagnóstico y trazabilidad

- **JSON (`json`)**: será utilizado para almacenar resultados de validaciones y diagnósticos.
- **Hashlib (`hashlib`)**: permitirá generar identificadores hash para comprobaciones de integridad y trazabilidad.
- **Warnings (`warnings`)**: facilita la gestión controlada de advertencias durante el procesamiento.

Estas dependencias conforman la base sobre la cual se desarrollarán los análisis de calidad de datos y los procesos de integración posteriores.

### Configuración del entorno de inspección

Adicionalmente, se ajustan algunas opciones de visualización de Pandas para facilitar la exploración de los datos.

Estas configuraciones permiten:

- Visualizar un mayor número de columnas simultáneamente.
- Inspeccionar más registros durante las etapas de diagnóstico.
- Presentar valores numéricos utilizando formatos homogéneos.

El objetivo es mejorar la legibilidad de los resultados y facilitar la interpretación de los diagnósticos que se generarán a lo largo del proceso.

### Detección automática de la raíz del proyecto

Uno de los principios de este proyecto es minimizar las dependencias respecto a la ubicación desde la cual se ejecuta el notebook.

Por esta razón, en lugar de asumir una ruta fija, se realiza una búsqueda ascendente a partir del directorio de trabajo actual hasta localizar una carpeta que contenga simultáneamente:

- El directorio `data`
- El directorio `notebooks`

Esta validación permite identificar de manera fiable la raíz del proyecto, independientemente del entorno de ejecución utilizado por cada integrante del equipo.

Si dicha estructura no puede localizarse, el proceso se detiene mediante una excepción controlada, evitando que operaciones posteriores se ejecuten sobre rutas incorrectas.

### Definición de directorios de trabajo

Una vez identificada la raíz del proyecto, se configuran los principales directorios que participarán en esta fase:

#### Directorio de datos originales

```text
data/raw/ml-32m/
```

Contiene los archivos originales obtenidos durante la fase de adquisición y que permanecerán inalterados durante todo el proyecto.

Estos archivos se consideran la fuente única de verdad y constituyen el punto de partida para cualquier reprocesamiento futuro.

#### Directorio de datos procesados

```text
data/processed/
```

Almacenará los resultados derivados de las actividades de limpieza, normalización e integración.

Entre los productos que se espera generar se encuentran:

- Tablas depuradas.
- Catálogos enriquecidos.
- Tablas puente.
- Tablas analíticas.
- Versiones derivadas utilizadas en análisis posteriores.

#### Directorio de diagnósticos

```text
data/processed/diagnostics/
```

Se destina al almacenamiento de métricas, reportes y resultados de control de calidad generados durante el proceso.

La separación explícita de estos diagnósticos facilita la trazabilidad de las decisiones tomadas durante la preparación de los datos.

### Garantía de reproducibilidad

Antes de continuar con la carga y validación de las tablas, el notebook verifica automáticamente la existencia de las carpetas de salida necesarias para almacenar los resultados generados en esta fase.

Si alguno de los directorios no existe, éste se crea automáticamente sin afectar la estructura existente del proyecto.

Esta estrategia garantiza que:

- El notebook pueda ejecutarse desde cero.
- Los resultados se almacenen en ubicaciones consistentes.
- Todos los integrantes del equipo compartan la misma organización de archivos.
- El flujo de trabajo sea completamente reproducible.

### Resultado esperado de esta etapa preliminar

Al finalizar esta fase de preparación del entorno, se dispone de una configuración estable y reproducible para iniciar los procedimientos de limpieza e integración.

Con ello se garantiza que las siguientes actividades:

- Evaluación de calidad de datos.
- Detección de inconsistencias.
- Identificación de valores faltantes.
- Análisis de duplicados.
- Normalización de variables.
- Construcción de tablas puente.
- Integración controlada de entidades.

se desarrollen sobre una estructura de trabajo homogénea, trazable y alineada con los principios metodológicos definidos para el proyecto.

In [1]:
from zipfile import ZipFile
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import shutil
import warnings

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Localizar la raíz del proyecto.
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "data").exists() and (p / "notebooks").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se localizó la raíz del proyecto. Ejecuta el notebook desde la carpeta "
        "'notebooks' o ajusta PROJECT_ROOT manualmente."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ml-32m"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DIAGNOSTICS_DIR = PROCESSED_DIR / "diagnostics"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Datos originales: {RAW_DIR}")
print(f"Datos procesados: {PROCESSED_DIR}")

Raíz del proyecto: /workspaces/Proyecto_Integrador_M1
Datos originales: /workspaces/Proyecto_Integrador_M1/data/raw/ml-32m
Datos procesados: /workspaces/Proyecto_Integrador_M1/data/processed


## Carga controlada de las tablas base

Una vez configurado el entorno de trabajo, se realiza la carga de las cuatro tablas principales de MovieLens 32M: `movies`, `ratings`, `tags` y `links`.

El objetivo es incorporar los datos preservando su estructura original, estableciendo tipos de datos consistentes y optimizando el consumo de memoria desde el inicio.

### Verificación de archivos

Antes de iniciar la lectura, se centralizan las rutas de los archivos originales y se verifica su disponibilidad en `data/raw/ml-32m/`.

Esta validación permite detectar archivos faltantes y detener la ejecución mediante `FileNotFoundError` antes de generar resultados parciales o inconsistentes.

### Estructura relacional

Cada tabla representa una unidad de observación diferente:

| Tabla | Unidad de observación |
|---|---|
| `movies` | Una película del catálogo |
| `ratings` | Una calificación de un usuario a una película |
| `tags` | Una etiqueta asignada por un usuario a una película |
| `links` | Identificadores externos asociados a una película |

Reconocer esta granularidad es fundamental para evitar interpretaciones incorrectas de los registros y prevenir duplicaciones artificiales durante las futuras integraciones.

### Tipificación y optimización de memoria

La lectura de los archivos utiliza tipos de datos explícitos mediante `dtype`, evitando conversiones innecesarias y reduciendo el consumo de memoria.

Esta estrategia también garantiza la compatibilidad de los identificadores compartidos entre tablas, facilitando las operaciones de unión y validación posteriores.

Se presta especial atención a `ratings`, debido a su volumen de aproximadamente 32 millones de registros. Para su procesamiento se priorizan tipos compactos y se consideran operaciones por bloques cuando el tamaño de las transformaciones lo requiere.

### Conservación de variables temporales

Las columnas `timestamp` de `ratings` y `tags` se conservan en su representación original como enteros Unix.

Esta decisión permite preservar la información de origen, mantener su trazabilidad y realizar conversiones temporales únicamente cuando sean necesarias para análisis específicos.

Las fechas legibles y otras variables derivadas se generarán posteriormente sin modificar los valores originales.

### Validación de la carga

Una vez finalizada la lectura, se verifica la correcta incorporación de las cuatro tablas mediante:

- Número de registros.
- Número de columnas.
- Tipos de datos asignados.

Esta comprobación permite confirmar la integridad básica de la carga y dimensionar las operaciones de limpieza e integración posteriores.

### Resultado de la fase

Se dispone de las cuatro tablas base de MovieLens 32M cargadas y preparadas para su procesamiento, con archivos previamente verificados, tipos de datos consistentes, identificadores compatibles y timestamps originales preservados.

Con esta estructura se inicia la siguiente etapa: el diagnóstico sistemático de calidad de los datos.

In [2]:
FILES = {
    "movies": RAW_DIR / "movies.csv",
    "ratings": RAW_DIR / "ratings.csv",
    "tags": RAW_DIR / "tags.csv",
    "links": RAW_DIR / "links.csv",
}

missing_files = [name for name, path in FILES.items() if not path.exists()]
if missing_files:
    print("Archivos fuente no encontrados. Iniciando descarga de MovieLens 32M...")
    zip_path = RAW_DIR.parent / "ml-32m.zip"
    ML32M_URL = "https://files.grouplens.org/datasets/movielens/ml-32m.zip"

    try:
        from urllib.request import urlretrieve
        # Descargar dataset localmente
        urlretrieve(ML32M_URL, zip_path)

        # Extraer contenido desde el archivo local ya descargado
        with ZipFile(zip_path, "r") as z:
            z.extractall(RAW_DIR.parent)

        # Eliminar el archivo zip para ahorrar espacio tras extraerlo
        zip_path.unlink()

        print("Dataset descargado y extraído correctamente.")

    except Exception as e:
        raise RuntimeError(
            f"No fue posible encontrar los archivos locales ni descargar el dataset: {e}"
        )


movies = pd.read_csv(
    FILES["movies"],
    encoding="utf-8",
    dtype={"movieId": "int32", "title": "string", "genres": "string"}
)

ratings = pd.read_csv(
    FILES["ratings"],
    encoding="utf-8",
    dtype={"userId": "int32", "movieId": "int32",
           "rating": "float32", "timestamp": "int64"}
)

tags = pd.read_csv(
    FILES["tags"],
    encoding="utf-8",
    dtype={"userId": "int32", "movieId": "int32",
           "tag": "string", "timestamp": "int64"}
)

links = pd.read_csv(
    FILES["links"],
    encoding="utf-8",
    dtype={"movieId": "int32", "imdbId": "Int64", "tmdbId": "Int64"}
)

for name, df in {
    "movies": movies, "ratings": ratings, "tags": tags, "links": links
}.items():
    print(f"{name:>8}: {len(df):>12,} filas | {len(df.columns)} columnas")

  movies:       87,585 filas | 3 columnas
 ratings:   32,000,204 filas | 4 columnas
    tags:    2,000,072 filas | 4 columnas
   links:       87,585 filas | 3 columnas


## Validación de estructura y calidad de datos

Antes de iniciar la limpieza y transformación, se realiza un diagnóstico inicial de los conjuntos de datos de MovieLens. El objetivo es conocer su estructura, identificar posibles problemas de calidad y establecer una línea base que permita justificar las decisiones de procesamiento posteriores.

Esta etapa es exclusivamente exploratoria: **no se modifican registros ni se aplican tratamientos sobre los datos originales.**

### Diagnóstico general por tabla

Para cada dataset se calculan los siguientes indicadores:

- **Dimensiones:** número de registros y columnas, para conocer el volumen y la estructura de la información.
- **Valores únicos:** diversidad de valores observados en las variables.
- **Duplicados completos:** cantidad de filas idénticas en todas sus columnas.
- **Valores nulos:** número total de celdas sin información.
- **Completitud:** porcentaje de valores disponibles respecto al total de celdas.
- **Consumo de memoria:** espacio requerido por cada tabla en Pandas, relevante para evaluar la eficiencia computacional.

Estos indicadores permiten obtener una visión general de la calidad estructural de cada conjunto de datos y detectar posibles anomalías.

### Diagnóstico por variable

Complementariamente, se realiza una inspección individual de las columnas para identificar características que podrían pasar inadvertidas en el análisis global.

Se consideran las siguientes métricas:

- **Tipo de dato:** verifica que cada atributo tenga una representación adecuada.
- **Valores no nulos:** cantidad de observaciones disponibles.
- **Valores nulos:** cantidad de observaciones sin información.
- **Porcentaje de nulos:** proporción de datos faltantes respecto al total de registros.
- **Cardinalidad:** número de valores distintos, útil para reconocer variables categóricas, identificar posibles identificadores y comprender la diversidad de los atributos.

### Consideraciones metodológicas

Es importante distinguir entre características propias de los datos y problemas reales de calidad:

- Una cardinalidad baja no implica necesariamente un problema; puede ser consecuencia de una escala o categoría definida.
- Los valores repetidos no representan necesariamente duplicados, ya que distintas observaciones pueden compartir los mismos atributos.
- Los valores nulos no siempre indican errores; pueden corresponder a información opcional o no disponible.
- Un porcentaje reducido de nulos puede ocultar una cantidad considerable de observaciones faltantes en tablas de gran tamaño.

Por lo tanto, las métricas obtenidas deben interpretarse considerando la naturaleza y la unidad de observación de cada tabla.

### Resultado de la evaluación

El diagnóstico establece una línea base del estado original de MovieLens y proporciona evidencia para orientar las siguientes etapas de procesamiento, particularmente en:

- Identificación y tratamiento de valores faltantes.
- Evaluación de registros duplicados.
- Verificación de tipos de datos.
- Validación de identificadores y relaciones entre tablas.
- Optimización de la representación y el almacenamiento de los datos.

De esta manera, las decisiones de limpieza e integración se fundamentan en los resultados observados y no en supuestos sobre la calidad de la información.

In [3]:
import gc

tables = {
    "movies": movies,
    "ratings": ratings,
    "tags": tags,
    "links": links
}

overview_rows = []
column_diagnostics = []

for name, df in tables.items():
    total_cells = df.shape[0] * df.shape[1]
    null_cells = int(df.isna().sum().sum())
    completeness = 100 * (1 - null_cells / total_cells) if total_cells else np.nan

    overview_rows.append({
        "dataset": name,
        "records": len(df),
        "columns": len(df.columns),
        "distinct_values_sum": int(df.nunique(dropna=True).sum()),
        "full_row_duplicates": int(df.duplicated().sum()),
        "null_cells": null_cells,
        "completeness_pct": completeness,
        "memory_mb": df.memory_usage(deep=True).sum() / (1024 ** 2)
    })

    for col in df.columns:
        nulls = int(df[col].isna().sum())
        n = len(df)
        column_diagnostics.append({
            "dataset": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null": int(df[col].notna().sum()),
            "nulls": nulls,
            "null_pct": (100 * nulls / n) if n else np.nan,
            "unique_non_null": int(df[col].nunique(dropna=True))
        })

    gc.collect()

overview = pd.DataFrame(overview_rows)
column_quality = pd.DataFrame(column_diagnostics)

display(overview.round({"completeness_pct": 5, "memory_mb": 2}))
display(column_quality.round({"null_pct": 6}))

: 

## Evaluación de llaves candidatas y detección de duplicados

Antes de integrar las tablas, es necesario identificar los atributos que permiten distinguir de manera única cada observación. Esta evaluación busca validar la granularidad de los datos y diferenciar los duplicados reales de las repeticiones legítimas del modelo relacional.

### Identificación de llaves candidatas

De acuerdo con la estructura de MovieLens, se establecen las siguientes llaves candidatas:

| Tabla | Llave candidata | Descripción |
|---|---|---|
| `movies` | `movieId` | Identifica cada película del catálogo. |
| `links` | `movieId` | Identifica la correspondencia de cada película con sistemas externos. |
| `ratings` | `(userId, movieId)` | Identifica la interacción entre un usuario y una película. |
| `tags` | `(userId, movieId, tag, timestamp)` | Permite identificar coincidencias exactas entre registros de etiquetas. |

En `ratings`, la combinación de usuario y película permite evaluar si existen múltiples calificaciones para una misma interacción.

En `tags`, se utiliza una combinación más amplia porque un usuario puede asignar distintas etiquetas a una película e incluso registrar eventos en diferentes momentos.

### Diagnóstico de unicidad

Para cada tabla se evalúa el comportamiento de las llaves propuestas mediante:

- **Registros totales:** número de observaciones.
- **Llaves distintas:** cantidad de combinaciones únicas.
- **Filas con llaves repetidas:** registros que participan en combinaciones duplicadas.

Estos indicadores permiten identificar posibles inconsistencias y comprobar si las llaves propuestas representan adecuadamente la granularidad de cada tabla.

### Detección de títulos repetidos

Adicionalmente, se inspecciona la variable `title` de `movies` para identificar películas que comparten el mismo título, pero tienen distintos identificadores.

Se realizan dos comparaciones:

1. **Coincidencia exacta:** agrupa los títulos tal como aparecen en los datos originales.
2. **Coincidencia normalizada:** aplica `strip()` y `casefold()` para eliminar diferencias de espacios en los extremos y capitalización.

La normalización se utiliza exclusivamente con fines diagnósticos, sin modificar los títulos originales.

### Criterio de conservación de identidad

La coincidencia textual no es suficiente para determinar que dos películas representan la misma entidad. Distintas producciones pueden compartir un título debido a remakes, adaptaciones o coincidencias nominales.

Por tanto, se establece el siguiente criterio:

> La identidad de cada película se determina mediante `movieId`, no mediante la coincidencia de su título.

En esta fase no se eliminan ni consolidan registros por similitud textual. Los resultados únicamente permiten identificar casos que podrían requerir una revisión posterior.

### Resultado de la evaluación

Esta etapa permite documentar la unicidad de las llaves candidatas, comprender la granularidad de cada tabla e identificar posibles duplicidades antes de realizar las integraciones.

Los hallazgos obtenidos servirán como referencia para validar las relaciones entre tablas y prevenir uniones que generen observaciones artificialmente duplicadas.

In [ ]:
def key_report(df, key_columns, table_name):
    duplicate_mask = df.duplicated(subset=key_columns, keep=False)
    return {
        "table": table_name,
        "key": " + ".join(key_columns),
        "rows": len(df),
        "duplicated_key_rows": int(duplicate_mask.sum()),
        "distinct_key_combinations": int(df[key_columns].drop_duplicates().shape[0])
    }

key_checks = pd.DataFrame([
    key_report(movies, ["movieId"], "movies"),
    key_report(links, ["movieId"], "links"),
    key_report(ratings, ["userId", "movieId"], "ratings"),
    key_report(tags, ["userId", "movieId", "tag", "timestamp"], "tags")
])
display(key_checks)

# Diagnóstico de títulos exactos repetidos.
title_counts = (
    movies.groupby("title", dropna=False)
    .agg(record_count=("movieId", "size"),
         movie_ids=("movieId", lambda s: list(s)))
    .query("record_count > 1")
    .sort_values(["record_count", "title"], ascending=[False, True])
    .reset_index()
)

# Normalización temporal para localizar coincidencias de escritura.
movies["title_norm"] = (
    movies["title"].astype("string").str.strip().str.casefold()
)

normalized_title_counts = (
    movies.groupby("title_norm", dropna=False)
    .agg(record_count=("movieId", "size"),
         titles=("title", lambda s: list(s)),
         movie_ids=("movieId", lambda s: list(s)))
    .query("record_count > 1")
    .sort_values("record_count", ascending=False)
    .reset_index()
)

print("Grupos de títulos exactos repetidos:", len(title_counts))
print("Grupos repetidos tras normalización:", len(normalized_title_counts))
display(title_counts.head(30))

,table,key,rows,duplicated_key_rows,distinct_key_combinations
0,movies,movieId,87585,0,87585
1,links,movieId,87585,0,87585
2,ratings,userId + movieId,32000204,0,32000204
3,tags,userId + movieId + tag + timestamp,2000072,0,2000072


Grupos de títulos exactos repetidos: 198
Grupos repetidos tras normalización: 200


,title,record_count,movie_ids
0,Alone (2020),4,"[212511, 223191, 224463, 273117]"
1,Journey to the Center of the Earth (2008),3,"[60514, 192003, 254480]"
2,The Box (2021),3,"[246860, 252868, 276057]"
3,The Plague (2006),3,"[128255, 128261, 206925]"
4,2020 (2020),2,"[230389, 245234]"
5,ATM (2012),2,"[96456, 217057]"
6,Absolution (2015),2,"[136696, 151797]"
7,Adam (2019),2,"[209761, 223916]"
8,Aladdin (1992),2,"[588, 114240]"
9,Alone (2017),2,"[181565, 198719]"


In [ ]:
dup = movies[movies.duplicated("title", keep=False)].merge(links, on="movieId", how="left")
grupos = dup.groupby("title").agg(
    n_ids=("movieId", "nunique"),
    n_imdb=("imdbId", "nunique"),
    n_tmdb=("tmdbId", "nunique"),
)
candidatos = grupos[(grupos.n_imdb == 1) | (grupos.n_tmdb == 1)]
len(grupos), len(candidatos)

(198, 20)

In [ ]:
grupos[(grupos.n_imdb == 1) | (grupos.n_tmdb == 1)]

,n_ids,n_imdb,n_tmdb
title,,,
Berlin Calling (2008),2,2,1
Clear History (2013),2,2,1
Clockstoppers (2002),2,2,1
Confessions of a Dangerous Mind (2002),2,2,1
Enron: The Smartest Guys in the Room (2005),2,2,1
Eros (2004),2,2,1
God's Angry Man (1980),2,2,1
Good Boys (2019),2,2,1
Lagaan: Once Upon a Time in India (2001),2,2,1


In [ ]:
movies.loc[movies["title"].eq("Berlin Calling (2008)")]

,movieId,title,genres,title_norm
13084,66511,Berlin Calling (2008),Comedy|Drama,berlin calling (2008)
56098,191775,Berlin Calling (2008),Comedy|Drama,berlin calling (2008)


In [ ]:
links[links["movieId"].isin([66511, 191775])]

In [ ]:
df_aux=ratings[ratings["movieId"].isin([66511, 191775])]
df_aux.groupby("movieId").agg(user_count=("userId", "nunique"))

,user_count
movieId,
66511,98
191775,24


## Cobertura del catálogo y actividad por película

Una vez evaluada la estructura relacional, se analiza la participación de las películas del catálogo (`movies`) en las actividades registradas mediante calificaciones (`ratings`) y etiquetas (`tags`).

El objetivo es cuantificar la cobertura efectiva del catálogo, identificar películas sin actividad y detectar referencias que no correspondan a registros existentes.

### Cobertura de actividad

La evaluación distingue cuatro conjuntos de películas:

- **Catálogo:** películas registradas en `movies`.
- **Calificaciones:** películas con al menos una observación en `ratings`.
- **Etiquetas:** películas con al menos una observación en `tags`.
- **Identificadores externos:** películas con correspondencias registradas en `links`.

Para evitar contabilizar varias veces una misma película, las comparaciones se realizan mediante identificadores únicos (`movieId`), independientemente del número de interacciones registradas.

La cobertura de calificaciones y etiquetas se calcula respecto al catálogo:

$$
\text{Cobertura}(\%) =
\frac{\text{Películas con actividad}}{\text{Películas en el catálogo}}
\times 100
$$

### Validación de integridad referencial

Adicionalmente, se comprueba la correspondencia de los identificadores entre las tablas.

Se consideran los siguientes escenarios:

- **Películas sin calificaciones:** registros del catálogo que no aparecen en `ratings`. Es una situación válida.
- **Películas sin etiquetas:** registros sin actividad en `tags`. También es válido, dado el carácter opcional de las etiquetas.
- **Películas sin identificadores externos:** registros de `movies` sin correspondencia en `links`, lo cual no implica necesariamente un problema.
- **Referencias inexistentes:** identificadores presentes en `ratings` o `tags` que no existen en `movies`. Estos casos representan posibles inconsistencias de integridad referencial.

### Resultado de la evaluación

Este análisis permite cuantificar el nivel de utilización del catálogo, identificar películas sin actividad registrada y verificar la consistencia de las referencias entre tablas.

Los resultados servirán para orientar las posteriores integraciones y distinguir entre ausencia legítima de actividad y posibles anomalías en los identificadores.

In [ ]:
catalog_movie_ids = set(movies["movieId"].dropna().unique())
rating_movie_ids = set(ratings["movieId"].dropna().unique())
tag_movie_ids = set(tags["movieId"].dropna().unique())
link_movie_ids = set(links["movieId"].dropna().unique())

coverage = pd.DataFrame([
    {
        "source": "ratings",
        "distinct_movie_ids": len(rating_movie_ids),
        "catalog_movies_covered": len(catalog_movie_ids & rating_movie_ids),
        "catalog_movies_without_activity": len(catalog_movie_ids - rating_movie_ids),
        "ids_not_in_catalog": len(rating_movie_ids - catalog_movie_ids),
        "catalog_coverage_pct": 100 * len(catalog_movie_ids & rating_movie_ids) / len(catalog_movie_ids)
    },
    {
        "source": "tags",
        "distinct_movie_ids": len(tag_movie_ids),
        "catalog_movies_covered": len(catalog_movie_ids & tag_movie_ids),
        "catalog_movies_without_activity": len(catalog_movie_ids - tag_movie_ids),
        "ids_not_in_catalog": len(tag_movie_ids - catalog_movie_ids),
        "catalog_coverage_pct": 100 * len(catalog_movie_ids & tag_movie_ids) / len(catalog_movie_ids)
    },
    {
        "source": "links",
        "distinct_movie_ids": len(link_movie_ids),
        "catalog_movies_covered": len(catalog_movie_ids & link_movie_ids),
        "catalog_movies_without_activity": len(catalog_movie_ids - link_movie_ids),
        "ids_not_in_catalog": len(link_movie_ids - catalog_movie_ids),
        "catalog_coverage_pct": 100 * len(catalog_movie_ids & link_movie_ids) / len(catalog_movie_ids)
    }
])

display(coverage.round(4))

movies_without_ratings = movies.loc[
    ~movies["movieId"].isin(rating_movie_ids), ["movieId", "title", "genres"]
].copy()

movies_without_tags = movies.loc[
    ~movies["movieId"].isin(tag_movie_ids), ["movieId", "title", "genres"]
].copy()

print("Películas sin calificaciones:", len(movies_without_ratings))
print("Películas sin etiquetas:", len(movies_without_tags))

,source,distinct_movie_ids,catalog_movies_covered,catalog_movies_without_activity,ids_not_in_catalog,catalog_coverage_pct
0,ratings,84432,84432,3153,0,96.4001
1,tags,51323,51323,36262,0,58.5979
2,links,87585,87585,0,0,100.0000


Películas sin calificaciones: 3153
Películas sin etiquetas: 36262


## Cobertura observada del catálogo

Los resultados obtenidos muestran que la integridad referencial del conjunto de datos es consistente.

### Cobertura mediante calificaciones

La tabla `ratings` contiene actividad asociada a:

- **84,432 películas distintas**.
- Lo anterior representa aproximadamente **96.40 %** del catálogo completo.

Esto implica que únicamente:

- **3,153 películas** del catálogo no han recibido ninguna calificación registrada.

Desde una perspectiva analítica, este resultado sugiere una cobertura extremadamente alta del catálogo por parte de los usuarios.

### Cobertura mediante etiquetas

La tabla `tags` presenta una cobertura menor, algo esperado debido al carácter opcional de las etiquetas.

Los resultados muestran:

- **51,323 películas etiquetadas**.
- Una cobertura aproximada de **58.60 %** del catálogo.

En consecuencia:

- **36,262 películas** no presentan etiquetas registradas.

Este comportamiento no debe interpretarse como una deficiencia de calidad, sino como una característica natural de la actividad de etiquetado, la cual requiere una participación más activa por parte de los usuarios.

### Cobertura mediante links

La tabla `links` presenta la cobertura más completa.

Los resultados indican que:

- Las **87,585 películas** del catálogo poseen correspondencia en la tabla de identificadores externos.
- La cobertura alcanza el **100 %**.

Este resultado confirma que todas las películas del catálogo pueden vincularse con sistemas externos como IMDb o TMDb.

## Verificación de consistencia referencial

Uno de los hallazgos más importantes de este análisis es que ninguna de las tres tablas relacionadas presenta identificadores de películas inexistentes en el catálogo principal.

Es decir:

- No existen `movieId` en `ratings` que no estén presentes en `movies`.
- No existen `movieId` en `tags` que no estén presentes en `movies`.
- No existen `movieId` en `links` que no estén presentes en `movies`.

La ausencia de estas discrepancias constituye una evidencia sólida de integridad referencial dentro del conjunto de datos.

## Identificación de películas sin actividad

Finalmente, se generan subconjuntos específicos que contienen las películas que no presentan actividad registrada.

Estas tablas conservan información básica de cada película:

- `movieId`
- `title`
- `genres`

y pueden utilizarse posteriormente para:

- Investigar características de las películas sin interacción.
- Comparar películas activas frente a inactivas.
- Evaluar sesgos de participación dentro de la comunidad de usuarios.
- Analizar la distribución de géneros o periodos de lanzamiento en películas sin actividad.

## Interpretación general

Los resultados obtenidos muestran que el catálogo MovieLens presenta una estructura altamente consistente.

Los principales hallazgos son:

- La cobertura de calificaciones alcanza más del 96 % del catálogo.
- Las etiquetas cubren aproximadamente el 59 % de las películas disponibles.
- La tabla de identificadores externos cubre la totalidad del catálogo.
- No se detectan rupturas de integridad referencial entre las tablas analizadas.
- Las películas sin actividad representan una fracción relativamente pequeña del conjunto total.

En conjunto, estos resultados proporcionan evidencia de que las relaciones entre tablas son consistentes y que el proceso de integración podrá realizarse utilizando `movieId` como llave principal de forma segura y controlada.

## Análisis de valores nulos y evaluación de completitud

Tras validar la estructura general de las tablas y confirmar la consistencia de los identificadores utilizados para la integración, el siguiente paso consiste en analizar detalladamente la presencia de valores faltantes.

Los valores nulos constituyen uno de los aspectos más relevantes dentro de cualquier proceso de calidad de datos. Sin embargo, su mera existencia no implica necesariamente un problema ni justifica de forma automática la eliminación o imputación de registros.

Por este motivo, antes de aplicar cualquier tratamiento, se realiza una evaluación específica orientada a comprender:

- Dónde aparecen los valores faltantes.
- Cuántos registros están afectados.
- Qué significado tendría eliminar o modificar dichas observaciones.
- Cómo impactan sobre la integridad del modelo de datos.

## Identificación sistemática de valores faltantes

Como primer paso, se realiza un recorrido por todas las columnas de las tablas analizadas con el objetivo de identificar aquellas variables que contienen valores nulos.

Para cada caso se calculan dos métricas complementarias:

### Número absoluto de nulos

Representa la cantidad exacta de registros afectados dentro de una variable.

Esta medida permite dimensionar directamente el volumen de información faltante.

### Porcentaje de nulidad

Corresponde a la proporción de registros nulos respecto al tamaño total de la tabla.

A diferencia de la etapa de caracterización general, en esta ocasión el porcentaje se reporta con un mayor nivel de precisión para evitar interpretaciones erróneas.

Esto resulta especialmente importante en conjuntos de datos de gran tamaño, donde cantidades aparentemente pequeñas de registros pueden desaparecer visualmente al redondear los porcentajes.

Por ejemplo, una variable puede contener varios registros nulos y aun así mostrar un porcentaje cercano a cero cuando el tamaño total de la tabla es muy elevado.

In [ ]:
tables = {
    "movies": movies,
    "ratings": ratings,
    "tags": tags,
    "links": links
}
null_detail = []
for table_name, df in tables.items():
    for col in df.columns:
        n_null = int(df[col].isna().sum())
        if n_null:
            null_detail.append({
                "table": table_name,
                "column": col,
                "null_count": n_null,
                "null_pct": 100 * n_null / len(df)
            })

null_report = pd.DataFrame(null_detail)
display(null_report.round({"null_pct": 8}))

# Inspección de registros con etiqueta nula.
tags_null = tags.loc[tags["tag"].isna()].copy()
display(tags_null.head(30))

# Inspección de películas sin identificador TMDB.
tmdb_missing = links.loc[links["tmdbId"].isna()].copy()
display(tmdb_missing.head(30))

NameError: name 'tables' is not defined

## Resultados identificados

El análisis muestra que únicamente dos variables presentan valores faltantes:

| Tabla | Variable | Registros nulos |
|---------|---------|---------|
| `tags` | `tag` | 17 |
| `links` | `tmdbId` | 124 |

No se detectan valores faltantes en los identificadores principales utilizados para la integración de tablas.

Este resultado es particularmente importante porque garantiza que las relaciones basadas en `movieId` y `userId` pueden construirse sin pérdida de información derivada de claves incompletas.

## Análisis de los registros con etiquetas faltantes

La tabla `tags` contiene únicamente **17 registros** cuyo contenido de etiqueta (`tag`) está ausente.

Considerando el volumen total de observaciones presentes en la tabla, este nivel de nulidad resulta extremadamente reducido.

### Inspección de los registros afectados

Antes de decidir cualquier acción correctiva, se inspeccionan directamente los registros que presentan ausencia de texto en la columna `tag`.

La revisión permite verificar que:

- Los identificadores de usuario permanecen disponibles.
- Los identificadores de película se encuentran completos.
- El momento de creación de la etiqueta sigue registrado mediante `timestamp`.
- La única información ausente corresponde al contenido textual de la etiqueta.

Por tanto, el evento de etiquetado sigue existiendo dentro del sistema, aunque no se disponga del texto asociado.

### Decisión adoptada

No se sustituyen los valores faltantes por cadenas artificiales como:

```text
"unknown"
```

o

```text
"sin_etiqueta"
```

ya que dichas categorías no forman parte de la información original distribuida por MovieLens.

Asimismo, los registros no se eliminan de la tabla base.

Su tratamiento quedará restringido a análisis específicos que requieran necesariamente el contenido textual de las etiquetas.

## Análisis de identificadores TMDB faltantes

La segunda fuente de nulidad identificada corresponde a la variable:

```text
tmdbId
```

dentro de la tabla `links`.

Se detectan **124 películas** sin correspondencia en The Movie Database (TMDb).

### Evaluación del impacto

La ausencia de un identificador TMDb no implica la pérdida de la película dentro del ecosistema MovieLens.

Cada uno de estos registros mantiene disponible:

- `movieId`
- `imdbId`

Por tanto, la película continúa siendo identificable tanto dentro del catálogo interno como mediante referencias externas alternativas.

### Interpretación

Existen múltiples razones legítimas por las cuales una película podría carecer de identificador TMDb:

- Ausencia temporal en dicha plataforma.
- Cambios en la catalogación histórica.
- Diferencias entre sistemas de referencia.
- Problemas de sincronización entre catálogos externos.

En cualquier caso, la falta de un `tmdbId` no compromete la integridad del modelo principal de datos.

### Decisión adoptada

Los registros afectados se conservan íntegramente.

La ausencia de un identificador externo no justifica la eliminación de películas del catálogo, especialmente cuando los identificadores principales continúan disponibles.

## Principio de conservación de datos

Siguiendo los criterios metodológicos definidos para este proyecto, se adopta una estrategia de preservación de la información original.

En consecuencia:

- No se eliminan registros debido a valores nulos.
- No se imputan categorías artificiales.
- No se reemplazan identificadores faltantes.
- No se modifica el contenido original de las tablas base.

Este enfoque permite mantener la trazabilidad completa de los datos distribuidos por MovieLens.

## Implicaciones para análisis posteriores

Aunque los registros se conservan, es importante reconocer que ciertas actividades analíticas requerirán tratamientos específicos.

Por ejemplo:

- Los análisis de texto deberán excluir explícitamente registros donde `tag` sea nulo.
- Las integraciones con TMDb deberán limitarse a películas que dispongan de un `tmdbId` válido.
- Los procesos de enriquecimiento externo deberán documentar claramente las observaciones descartadas.

Estas exclusiones se realizarán únicamente en el contexto de cada análisis particular y nunca modificando las tablas base del proyecto.

## Conclusiones

La evaluación de completitud indica que el conjunto MovieLens 32M presenta niveles muy altos de calidad estructural.

Los principales hallazgos son:

- La cantidad de valores nulos es extremadamente reducida respecto al volumen total de datos.
- No existen valores faltantes en los identificadores principales de integración.
- Los nulos detectados corresponden a variables secundarias y no comprometen la integridad relacional del modelo.
- La conservación de los registros permite mantener la trazabilidad completa de la información original.
- Cualquier tratamiento posterior sobre valores faltantes se realizará de forma contextual y documentada.

En consecuencia, no se aplicarán transformaciones destructivas sobre las tablas base y se continuará con las siguientes etapas de limpieza e integración preservando íntegramente la información original distribuida por MovieLens.

## Correspondencia de identificadores y análisis de actividad de usuarios

Después de verificar la calidad estructural de los datos y evaluar la completitud de los atributos, el siguiente paso consiste en validar la consistencia de los identificadores utilizados para relacionar las distintas tablas del ecosistema MovieLens.

Esta etapa tiene dos objetivos principales:

1. Comprobar que las relaciones entre películas mantienen integridad referencial.
2. Comprender cómo participan los usuarios en las distintas actividades registradas por la plataforma.

La correcta interpretación de estos identificadores resulta esencial para garantizar que las integraciones posteriores se realicen de forma segura y sin pérdida de información.

## Correspondencia de identificadores de películas

La variable `movieId` constituye el identificador interno principal del conjunto de datos MovieLens.

Este atributo aparece en las cuatro tablas analizadas:

- `movies`
- `ratings`
- `tags`
- `links`

Por lo tanto, representa la pieza central del modelo relacional y será la clave utilizada para la integración entre entidades a lo largo del proyecto.

### Validación de integridad referencial

Con el fin de verificar la consistencia del modelo, se comparan los conjuntos de identificadores presentes en:

- La tabla de catálogo (`movies`).
- Las tablas de actividad (`ratings` y `tags`).
- La tabla de identificadores externos (`links`).

Este breve análisis busca responder una pregunta fundamental:

> ¿Todos los identificadores de películas observados en las tablas dependientes existen realmente dentro del catálogo oficial de películas?

Para ello se calculan:

- Número de películas distintas observadas en cada tabla.
- Número de identificadores encontrados en el catálogo.
- Número de identificadores huérfanos o sin correspondencia.

In [ ]:
# Integridad referencial de movieId.
referential_report = pd.DataFrame([
    {
        "child_table": "ratings",
        "distinct_movie_ids": len(rating_movie_ids),
        "ids_found_in_movies": len(rating_movie_ids & catalog_movie_ids),
        "orphan_movie_ids": len(rating_movie_ids - catalog_movie_ids)
    },
    {
        "child_table": "tags",
        "distinct_movie_ids": len(tag_movie_ids),
        "ids_found_in_movies": len(tag_movie_ids & catalog_movie_ids),
        "orphan_movie_ids": len(tag_movie_ids - catalog_movie_ids)
    },
    {
        "child_table": "links",
        "distinct_movie_ids": len(link_movie_ids),
        "ids_found_in_movies": len(link_movie_ids & catalog_movie_ids),
        "orphan_movie_ids": len(link_movie_ids - catalog_movie_ids)
    }
])
display(referential_report)

rating_users = set(ratings["userId"].unique())
tag_users = set(tags["userId"].unique())

user_activity = pd.DataFrame([{
    "users_with_ratings": len(rating_users),
    "users_with_tags": len(tag_users),
    "users_in_both": len(rating_users & tag_users),
    "users_only_ratings": len(rating_users - tag_users),
    "users_only_tags": len(tag_users - rating_users),
    "union_users": len(rating_users | tag_users),
    "taggers_pct_of_raters": 100 * len(rating_users & tag_users) / len(rating_users)
}])
display(user_activity)

,child_table,distinct_movie_ids,ids_found_in_movies,orphan_movie_ids
0,ratings,84432,84432,0
1,tags,51323,51323,0
2,links,87585,87585,0


,users_with_ratings,users_with_tags,users_in_both,users_only_ratings,users_only_tags,union_users,taggers_pct_of_raters
0,200948,15848,15848,185100,0,200948,7.8866


### Resultados de cobertura e integridad referencial

Los resultados obtenidos muestran una correspondencia completa entre las películas registradas en las tablas de actividad y el catálogo principal.

| Tabla | Películas únicas | Resultado |
|---|---:|---|
| `ratings` | 84,432 | Todas existen en `movies`. |
| `tags` | 51,323 | Todas existen en `movies`. |
| `links` | 87,585 | Cobertura completa del catálogo. |

No se identifican referencias a películas inexistentes en `movies`, lo que confirma la integridad referencial de los identificadores `movieId` en las relaciones evaluadas.

### Identificadores externos

La tabla `links` incorpora identificadores provenientes de otras plataformas:

- **IMDb:** `imdbId`.
- **The Movie Database:** `tmdbId`.

Estos atributos permiten enriquecer el catálogo e incorporar información de fuentes externas. Sin embargo, se mantienen como identificadores complementarios: `movieId` continúa siendo la referencia principal de MovieLens.

### Correspondencia de usuarios por actividad

A diferencia de las películas, los usuarios únicamente aparecen en `ratings` y `tags`. Por ello, se comparan ambos conjuntos mediante valores únicos de `userId` para identificar los patrones de participación.

Los resultados son los siguientes:

| Tipo de actividad | Usuarios |
|---|---:|
| Usuarios que califican | 200,948 |
| Usuarios que etiquetan | 15,848 |
| Usuarios que realizan ambas actividades | 15,848 |
| Usuarios exclusivos de calificaciones | 185,100 |
| Usuarios exclusivos de etiquetas | 0 |

<text color="secondary" size="xs">La clasificación se basa en la intersección y diferencia de los conjuntos de usuarios de ambas tablas.</text>

### Comparación de usuarios por tipo de actividad
Se construyen conjuntos únicos de userId para responder las siguientes preguntas:

- ¿Cuántos usuarios emiten calificaciones?
- ¿Cuántos usuarios generan etiquetas?
- ¿Cuántos realizan ambas actividades?
- ¿Existen usuarios que únicamente etiqueten y nunca califiquen?

### Interpretación

Los resultados muestran que todos los usuarios que registran etiquetas también aparecen en `ratings`. Por tanto, el etiquetado constituye un subconjunto de la actividad de calificación.

Únicamente el **7.89 %** de los usuarios que califican películas participa también en el etiquetado, lo que evidencia una menor frecuencia de esta actividad.

Esta diferencia es coherente con la naturaleza de ambas interacciones: calificar representa una acción más sencilla, mientras que etiquetar requiere una participación adicional del usuario.

La ausencia de usuarios exclusivos de etiquetado y de referencias a películas inexistentes no representa una anomalía, sino un comportamiento consistente con las relaciones observadas en el conjunto de datos.

### Conclusiones

La evaluación permite establecer que:

- Las relaciones basadas en `movieId` presentan integridad referencial en las tablas analizadas.
- `imdbId` y `tmdbId` complementan la identificación de películas sin sustituir a `movieId`.
- La actividad de etiquetado corresponde a un subconjunto de los usuarios que califican.
- No se identifican referencias huérfanas en las relaciones de películas y usuarios evaluadas.

Estos resultados proporcionan una base consistente para avanzar hacia la integración y el enriquecimiento de la información.

## Validación de la escala de calificaciones

Una vez verificada la integridad estructural de las tablas y la consistencia de los identificadores utilizados en el modelo relacional, se procede a validar uno de los atributos más importantes del conjunto de datos: la variable `rating`.

Las calificaciones constituyen la principal fuente de información sobre las preferencias de los usuarios y serán utilizadas en múltiples análisis posteriores.

Por esta razón, antes de realizar cualquier agregación o integración analítica, resulta indispensable confirmar que todas las observaciones cumplen las reglas definidas por la documentación oficial de MovieLens.

## Escala documentada por MovieLens

Según la documentación del conjunto de datos, las calificaciones deben pertenecer a una escala discreta de diez niveles comprendida entre **0.5** y **5.0**, utilizando incrementos de medio punto.

La escala permitida es la siguiente:

```text
0.5
1.0
1.5
2.0
2.5
3.0
3.5
4.0
4.5
5.0
```

En consecuencia, cualquier valor fuera de este conjunto representaría una inconsistencia que debería ser investigada antes de continuar con el análisis.

## Verificación de valores observados

La primera validación consiste en identificar todos los valores distintos presentes en la columna `rating` y compararlos con los valores permitidos por la especificación oficial.

Esta comprobación permite responder preguntas fundamentales como:

- ¿Se encuentran presentes todos los niveles de la escala?
- ¿Existen valores inesperados?
- ¿Se produjeron errores durante la carga de los datos?
- ¿Hay observaciones que contradigan la documentación oficial?

Los resultados deberían mostrar que los valores observados coinciden exactamente con los diez niveles permitidos por MovieLens.

## Detección de registros fuera de escala

Posteriormente se realiza una validación registro por registro para identificar posibles observaciones cuyo valor de calificación no pertenezca al conjunto permitido.

Esta comprobación resulta más robusta que simplemente contar valores únicos, ya que garantiza que cada observación individual cumple las reglas definidas por el sistema de calificación.

In [ ]:
allowed_ratings = np.arange(0.5, 5.0 + 0.5, 0.5)
observed_ratings = np.sort(ratings["rating"].dropna().unique())
invalid_rating_mask = ~ratings["rating"].isin(allowed_ratings)

rating_validation = {
    "allowed_values": allowed_ratings.tolist(),
    "observed_values": observed_ratings.tolist(),
    "invalid_rows": int(invalid_rating_mask.sum()),
    "minimum": float(ratings["rating"].min()),
    "maximum": float(ratings["rating"].max())
}
display(pd.Series(rating_validation, name="value"))

rating_distribution = (
    ratings["rating"].value_counts(dropna=False)
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="frequency")
)
rating_distribution["percentage"] = (
    100 * rating_distribution["frequency"] / len(ratings)
)
display(rating_distribution)

if invalid_rating_mask.any():
    invalid_ratings = ratings.loc[invalid_rating_mask].copy()
    invalid_ratings.to_csv(
        DIAGNOSTICS_DIR / "ratings_fuera_de_escala.csv",
        index=False, encoding="utf-8"
    )
    warnings.warn("Se detectaron calificaciones fuera de la escala documentada.")
else:
    print("Validación satisfactoria: todas las calificaciones pertenecen a la escala.")

allowed_values     [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, ...
observed_values    [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, ...
invalid_rows                                                       0
minimum                                                       0.5000
maximum                                                       5.0000
Name: value, dtype: object

,rating,frequency,percentage
0,0.5000,525132,1.6410
1,1.0000,946675,2.9583
2,1.5000,531063,1.6596
3,2.0000,2028622,6.3394
4,2.5000,1685386,5.2668
5,3.0000,6054990,18.9217
6,3.5000,4290105,13.4065
7,4.0000,8367654,26.1488
8,4.5000,2974000,9.2937
9,5.0000,4596577,14.3642


Validación satisfactoria: todas las calificaciones pertenecen a la escala.


## Resultado obtenido

La evaluación identifica:

```text
Registros fuera de escala: 0
```

Esto significa que ninguna calificación registrada infringe las restricciones documentadas por MovieLens.

Por tanto:

- No existen valores inválidos.
- No se detectan errores de captura evidentes.
- No hay indicios de corrupción en la variable `rating`.

### Validación de límites mínimos y máximos

Como comprobación adicional se analizan los valores extremos observados dentro de la distribución.

Los resultados muestran:

| Indicador | Valor |
|------------|------------|
| Calificación mínima | 0.5 |
| Calificación máxima | 5.0 |

Estos límites coinciden exactamente con la escala oficial del dataset.

La coincidencia entre los valores extremos observados y los límites documentados refuerza la evidencia de que la variable fue registrada correctamente.

### Distribución de frecuencias de las calificaciones

Además de validar la escala, se analiza la frecuencia con la que aparece cada nivel de calificación dentro del conjunto de datos.

Esta evaluación proporciona una primera aproximación al comportamiento general de los usuarios y permite detectar posibles anomalías en la distribución.

### Principales hallazgos

Los resultados muestran que las puntuaciones positivas dominan claramente la distribución.

Las categorías más frecuentes corresponden a:

| Calificación | Frecuencia aproximada |
|--------------|--------------|
| 4.0 | 8.37 millones |
| 3.0 | 6.05 millones |
| 5.0 | 4.60 millones |
| 3.5 | 4.29 millones |

Por el contrario, las puntuaciones más bajas presentan frecuencias considerablemente menores.

Por ejemplo:

- Las calificaciones de 0.5 representan aproximadamente el 1.64 % del total.
- Las calificaciones de 1.0 representan alrededor del 2.96 %.

### Interpretación preliminar

La distribución observada sugiere una tendencia positiva en las evaluaciones realizadas por los usuarios.

Este comportamiento es frecuente en sistemas de recomendación, donde los usuarios tienden a interactuar principalmente con contenidos que les resultan interesantes o satisfactorios.

No obstante, en esta etapa el análisis tiene únicamente un propósito de validación y descripción de la variable. La interpretación profunda de los patrones de valoración será desarrollada en las fases posteriores de análisis exploratorio.

## Estrategia ante posibles valores inválidos

Aunque no se detectaron registros fuera de escala, se establece explícitamente un protocolo de actuación para futuras revisiones de calidad.

Si se hubieran identificado observaciones inválidas:

- No se corregirían automáticamente.
- No se redondearían las calificaciones.
- No se modificarían los datos originales.

En su lugar:

1. Los registros serían aislados en un archivo específico de auditoría.
2. Se documentaría el incidente.
3. Se conservarían los datos originales para mantener la trazabilidad.
4. La decisión de tratamiento se tomaría posteriormente con evidencia suficiente.

Este enfoque evita introducir modificaciones arbitrarias en información cuyo origen o significado podría requerir una investigación adicional.

## Conclusiones

La validación realizada confirma que la variable `rating` cumple completamente las especificaciones documentadas por MovieLens.

Los principales resultados obtenidos son:

- Los valores observados coinciden exactamente con la escala oficial.
- No existen registros fuera de rango.
- Los límites mínimo y máximo corresponden a los definidos por la documentación.
- Se encuentran representados los diez niveles de calificación permitidos.
- No fue necesario aplicar correcciones ni transformaciones sobre la variable original.

En consecuencia, la columna `rating` puede considerarse una variable de alta calidad y apta para ser utilizada en los análisis descriptivos, estadísticos y de integración que se desarrollarán en las siguientes etapas del proyecto.

## Normalización de la variable `genres` y construcción de la tabla puente película–género

Una vez verificadas la calidad, completitud e integridad de las tablas principales, el siguiente paso consiste en abordar uno de los aspectos más importantes de la estructura del catálogo de películas: la representación de los géneros cinematográficos.

Esta etapa tiene como objetivo transformar una estructura desnormalizada en una representación relacional más adecuada para análisis posteriores, preservando en todo momento la información original distribuida por MovieLens.

La normalización de géneros permitirá realizar consultas, agregaciones y análisis comparativos de forma mucho más eficiente y consistente.

## Problema de modelado de la variable `genres`

En la tabla `movies`, los géneros de una película no se almacenan en registros independientes, sino dentro de una única celda de texto.

Por ejemplo:

```text
Action|Adventure|Sci-Fi
```

Desde una perspectiva descriptiva, este formato resulta compacto y fácilmente interpretable por una persona.

Sin embargo, desde el punto de vista analítico presenta diversas limitaciones.

### Limitaciones de la representación original

Cuando varios géneros se almacenan dentro de una misma celda:

- No es posible contar géneros directamente.
- Resulta difícil calcular estadísticas por género.
- Las agrupaciones requieren procesamiento adicional.
- Se complica la comparación entre categorías.
- Las integraciones con otras tablas requieren transformaciones previas.

Por ejemplo, responder preguntas como:

- ¿Cuántas películas pertenecen al género Acción?
- ¿Cuál es la valoración promedio de las películas de Ciencia Ficción?
- ¿Qué género recibe más etiquetas?
- ¿Qué género concentra más interacciones de usuarios?

requiere primero separar los géneros individuales contenidos en cada registro.

## Relación muchos a muchos entre películas y géneros

Conceptualmente, la relación existente entre películas y géneros corresponde a una relación de tipo muchos a muchos.

Esto significa que:

### Una película puede pertenecer a varios géneros

Por ejemplo:

```text
Toy Story
```

puede clasificarse simultáneamente como:

- Animation
- Adventure
- Comedy
- Children
- Fantasy

### Un género puede agrupar muchas películas

Por ejemplo:

```text
Drama
```

aparece asociado a miles de películas diferentes dentro del catálogo.

La estructura original de `movies` almacena esta relación comprimida en una única columna, dificultando su explotación analítica.

## Construcción de una tabla puente

Para representar adecuadamente esta relación se genera una tabla puente denominada:

```text
movie_genres
```

El objetivo es producir una estructura normalizada donde cada fila represente una única relación entre una película y un género.

La nueva estructura adopta la siguiente forma:

| movieId | genre |
|----------|----------|
| 1 | Adventure |
| 1 | Animation |
| 1 | Children |
| 1 | Comedy |
| 1 | Fantasy |

De esta manera, una película con cinco géneros generará cinco registros independientes dentro de la tabla puente.

## Conservación de la información original

La normalización no implica la eliminación de la información existente.

Antes de realizar cualquier transformación se conserva una copia de la columna original mediante:

```text
genres_original
```

Esta decisión responde a los principios metodológicos definidos para el proyecto:

- Preservar la información original.
- Mantener trazabilidad completa de las transformaciones.
- Permitir auditorías futuras.
- Facilitar la reproducción exacta del proceso de limpieza.

La columna original continúa disponible para referencias posteriores y para verificar cualquier transformación realizada.

## Proceso de expansión de géneros

La normalización se desarrolla mediante varias etapas:

### Separación de categorías

Cada cadena de texto se divide utilizando el carácter:

```text
|
```

que actúa como separador entre géneros.

### Expansión de registros

Posteriormente, cada género individual se transforma en una fila independiente mediante una operación de explosión de listas (`explode`).

### Limpieza básica

Se eliminan:

- Espacios residuales.
- Cadenas vacías.
- Posibles duplicados exactos de la combinación `(movieId, genre)`.

Como resultado se obtiene una representación limpia y normalizada de la relación película–género.

### Tratamiento de `(no genres listed)`

En la columna `genres`, MovieLens usa la etiqueta especial `(no genres listed)` para las películas a las que no se les asignó ningún género. Aunque aparece como texto, en la práctica representa un **valor faltante**.

Este faltante pasa inadvertido porque no está codificado como nulo: `isna()` no lo detecta, y si no se trata, el análisis lo consideraría un género más. Por eso, antes de continuar, cuantificamos cuántas películas y calificaciones involucra, analizamos su impacto sobre los resultados y, con base en eso, decidimos cómo tratarlas.

In [ ]:
SIN_GENERO = "(no genres listed)"

mask_sin_genero = movies["genres"].eq(SIN_GENERO)
print(mask_sin_genero.sum(), round(mask_sin_genero.mean() * 100, 2), "% de las películas")


7080 8.08 % de las películas


Del total de películas del catálogo, **7,080 (8.08%)** tienen la etiqueta `(no genres listed)`, es decir, no cuentan con ningún género asignado.

### Distribucion de los géneros considerando `(no genres listed)`

In [ ]:


movie_genres = (
    movies[["movieId", "genres"]]
    .assign(genre=lambda d: d["genres"].fillna("").str.split("|"))
    .explode("genre")
    [["movieId", "genre"]]
    .assign(genre=lambda d: d["genre"].astype("string").str.strip())
)

movie_genres = movie_genres.loc[
    movie_genres["genre"].notna() & movie_genres["genre"].ne("")
].drop_duplicates(["movieId", "genre"]).reset_index(drop=True)

genre_frequency = (
    movie_genres.groupby("genre")
    .agg(movie_count=("movieId", "nunique"))
    .sort_values("movie_count", ascending=False)
    .reset_index()
)
genre_frequency["catalog_pct"] = (
    100 * genre_frequency["movie_count"] / movies["movieId"].nunique()
)

print("Número de géneros distintos:", movie_genres["genre"].nunique())
print("Filas de la tabla puente:", len(movie_genres))
display(genre_frequency)


Número de géneros distintos: 20
Filas de la tabla puente: 154170


,genre,movie_count,catalog_pct
0,Drama,34175,39.0192
1,Comedy,23124,26.4018
2,Thriller,11823,13.4989
3,Romance,10369,11.8388
4,Action,9668,11.0384
5,Documentary,9363,10.6902
6,Horror,8654,9.8807
7,(no genres listed),7080,8.0836
8,Crime,6976,7.9648
9,Adventure,5402,6.1677


### Distribución de los géneros (incluyendo `(no genres listed)`)

Drama es el género más frecuente del catálogo: aparece en **34,175 películas (39.0%)**, seguido de Comedy con **23,124 (26.4%)** y Thriller con **11,823 (13.5%)**. Los siguientes géneros están muy cerca entre sí, con proporciones de entre 9.9% y 11.8% (Romance, Action, Documentary y Horror).

La categoría `(no genres listed)` ocupa el octavo lugar, con **7,080 películas (8.1%)**, una frecuencia comparable a la de géneros reales como Horror.

Los porcentajes **no suman 100%** porque una misma película puede tener varios géneros y se cuenta en cada uno de ellos. Estas cifras describen la composición del catálogo, no la cantidad de calificaciones por género.

### Imputación 
Se opta por un enfoque conservador, se imputara con la moda 

In [ ]:
SIN_GENERO = "(no genres listed)"
mask_sin_genero = movies["genres"].eq(SIN_GENERO)
genero_moda = (
    movies.loc[~mask_sin_genero, "genres"]
    .str.split("|").explode().str.strip()
    .value_counts().idxmax()
)
print("Moda:", genero_moda)
movies_imp = movies.copy()
movies_imp["genre_imputed"] = mask_sin_genero #se pone una etiqueta sobre el registro para indicar si es faltante o no
movies_imp.loc[mask_sin_genero, "genres"] = genero_moda

Moda: Drama


In [ ]:
movies_imp.head()

,movieId,title,genres,genres_original,genre_imputed
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure|Animation|Children|Comedy|Fantasy,False
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure|Children|Fantasy,False
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy|Romance,False
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy|Drama|Romance,False
4,5,Father of the Bride Part II (1995),Comedy,Comedy,False


In [ ]:
movie_genres_imp = (
    movies_imp[["movieId", "genres"]]
    .assign(genre=lambda d: d["genres"].fillna("").str.split("|"))
    .explode("genre")
    [["movieId", "genre"]]
    .assign(genre=lambda d: d["genre"].astype("string").str.strip())
)

genre_frequency_imp = (
    movie_genres_imp.groupby("genre")
    .agg(movie_count=("movieId", "nunique"))
    .sort_values("movie_count", ascending=False)
    .reset_index()
)
genre_frequency_imp["catalog_pct"] = (
    100 * genre_frequency_imp["movie_count"] / movies_imp["movieId"].nunique()
)

display(genre_frequency_imp)

,genre,movie_count,catalog_pct
0,Drama,41255,47.1028
1,Comedy,23124,26.4018
2,Thriller,11823,13.4989
3,Romance,10369,11.8388
4,Action,9668,11.0384
5,Documentary,9363,10.6902
6,Horror,8654,9.8807
7,Crime,6976,7.9648
8,Adventure,5402,6.1677
9,Sci-Fi,4907,5.6026


### Distribución de los géneros después de la imputación

Al imputar con la moda las 7,080 películas sin género, **todas** quedaron asignadas a Drama, que era el género más frecuente. Como consecuencia, Drama pasó de **34,175 películas (39.0%)** a **41,255 (47.1%)**, un aumento de 8.1 puntos porcentuales (cerca de 21% en términos relativos). Los demás géneros no cambian: Comedy se mantiene en 26.4% y Thriller en 13.5%.

El **orden de los géneros se conserva** (Drama, Comedy, Thriller, Romance, Action...), pero la brecha entre Drama y Comedy se ensancha de unos 12.6 a unos 20.7 puntos. La imputación, por tanto, no altera el ranking, pero sí **infla la presencia de Drama** en el catálogo 


## Beneficios analíticos de la normalización

La construcción de la tabla puente proporciona una estructura significativamente más adecuada para el análisis exploratorio y descriptivo.

Gracias a esta normalización será posible:

- Contar películas por género de forma directa.
- Calcular calificaciones promedio por categoría.
- Analizar actividad de usuarios por género.
- Estudiar distribución de etiquetas según categorías cinematográficas.
- Construir visualizaciones comparativas.
- Integrar géneros con métricas agregadas de películas.

Todo ello sin alterar ni perder la representación original almacenada en la tabla `movies`.

## Conclusiones

La normalización de la variable `genres` constituye una de las transformaciones estructurales más importantes del proyecto.

Los principales resultados obtenidos son:

- Se preserva la columna original de géneros.
- Se construye una tabla puente normalizada película–género.
- Se identifican 20 géneros distintos en el catálogo.
- Se generan 154,170 relaciones película–género.
- Se habilita una estructura adecuada para futuras agregaciones e integraciones analíticas.


## Construcción de métricas agregadas por película

Una vez verificadas las relaciones entre tablas, evaluada la cobertura del catálogo y normalizada la información de géneros, se construyen métricas agregadas a nivel de película.

El objetivo es enriquecer el catálogo con indicadores de actividad y comportamiento de los usuarios, **sin modificar su granularidad original**: una fila por película.

Las tablas `ratings` y `tags` contienen millones de eventos individuales, mientras que `movies` mantiene un registro único por película. Por ello, antes de integrar estas fuentes, es necesario resumir sus registros mediante agrupaciones por `movieId`.

## Motivación de los agregados

MovieLens utiliza una estructura transaccional en la que:

- Cada fila de `ratings` representa una calificación individual.
- Cada fila de `tags` representa una aplicación de etiqueta realizada por un usuario.

En consecuencia, una película puede aparecer numerosas veces en ambas tablas. Integrarlas directamente con `movies` podría provocar:

- Multiplicación artificial de filas.
- Incremento innecesario del volumen de datos.
- Relaciones many-to-many no deseadas.
- Distorsión de las estadísticas posteriores.

Para evitar estos problemas, se generan tablas resumen que concentran la actividad de cada película en un único registro.

## Agregación de calificaciones

La tabla `ratings` es la principal fuente de información sobre las valoraciones de los usuarios. Se construyen indicadores descriptivos agrupados por `movieId`, de modo que cada película tenga una sola fila en `rating_agg`.

| Métrica | Descripción e interpretación |
|---|---|
| `rating_count` | Número total de calificaciones recibidas. Funciona como indicador de popularidad y participación de los usuarios. |
| `rating_mean` | Calificación promedio. Permite estudiar la valoración general de cada película y las preferencias del público. |
| `rating_std` | Desviación estándar de las calificaciones. Una dispersión elevada sugiere mayor desacuerdo entre usuarios; una baja indica opiniones más homogéneas. |
| `rating_min` | Calificación mínima registrada para la película. |
| `rating_max` | Calificación máxima registrada para la película. |
| `rating_users` | Número de usuarios distintos que calificaron la película. Complementa el conteo total al distinguir la participación individual de los posibles registros repetidos. |

Estas métricas permiten estudiar tanto el nivel de aceptación de una película como el grado de consenso o discrepancia entre sus evaluaciones.

## Agregación de etiquetas

La tabla `tags` contiene información semántica generada por los usuarios mediante etiquetas descriptivas. Se construye un resumen agrupado por `movieId`, almacenado en `tag_agg`.

| Métrica | Descripción e interpretación |
|---|---|
| `tag_application_count` | Número total de aplicaciones de etiquetas registradas para la película. Cada evento se contabiliza como una observación independiente. |
| `non_null_tag_count` | Número de aplicaciones que contienen un valor textual no nulo. Permite distinguir los registros con información descriptiva válida, considerando los valores nulos detectados durante la evaluación de calidad. |
| `distinct_tags` | Número de etiquetas distintas asociadas a la película. Refleja la diversidad de términos utilizados por la comunidad para describirla. |
| `tag_users` | Número de usuarios distintos que han etiquetado la película. Proporciona una medida complementaria de participación. |

Estas métricas permiten explorar la riqueza descriptiva del catálogo y la participación de los usuarios en la caracterización de las películas.

## Preservación de la granularidad y validación

Un principio fundamental de esta etapa es evitar modificaciones accidentales en la granularidad de los datos.

Por ello, tanto `rating_agg` como `tag_agg` deben cumplir la siguiente condición:

> Cada tabla agregada debe contener como máximo una fila por película (`movieId`).

La validación posterior debe comprobar que no existan identificadores repetidos en las tablas agregadas y que su integración con `movies` no incremente el número de películas del catálogo.

De esta manera, los indicadores de calificaciones y etiquetas podrán incorporarse al catálogo sin generar duplicaciones ni distorsionar los análisis posteriores.

In [ ]:
rating_agg = (
    ratings.groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        rating_mean=("rating", "mean"),
        rating_std=("rating", "std"),
        rating_min=("rating", "min"),
        rating_max=("rating", "max"),
        rating_users=("userId", "nunique")
    )
    .reset_index()
)

tag_agg = (
    tags.groupby("movieId")
    .agg(
        tag_application_count=("tag", "size"),
        non_null_tag_count=("tag", "count"),
        distinct_tags=("tag", "nunique"),
        tag_users=("userId", "nunique")
    )
    .reset_index()
)

print("Filas en agregado de ratings:", len(rating_agg))
print("Filas en agregado de tags:", len(tag_agg))
print("¿rating_agg tiene movieId único?:", rating_agg["movieId"].is_unique)
print("¿tag_agg tiene movieId único?:", tag_agg["movieId"].is_unique)

Filas en agregado de ratings: 84432
Filas en agregado de tags: 51323
¿rating_agg tiene movieId único?: True
¿tag_agg tiene movieId único?: True


### Resultado de la validación

Las tablas agregadas presentan los siguientes resultados:

| Tabla | Películas registradas | Unicidad de `movieId` |
|---|---:|---|
| `rating_agg` | 84,432 | Cumplida |
| `tag_agg` | 51,323 | Cumplida |

La validación mediante `is_unique` confirma que no existen identificadores de película repetidos:

```python
rating_agg["movieId"].is_unique  # True
tag_agg["movieId"].is_unique     # True
```

Por tanto, ambas estructuras cumplen el requisito de unicidad y pueden integrarse con el catálogo mediante relaciones uno a uno, sin multiplicar sus registros.

## Interpretación de valores faltantes en métricas agregadas

La ausencia de actividad debe distinguirse de la ausencia de un valor calculable.

Por ejemplo, una película sin calificaciones debe presentar:

```text
rating_count = 0
rating_mean = NaN
```

Mientras que `rating_count` puede tomar el valor cero, `rating_mean` debe permanecer como `NaN`, ya que no existen observaciones para calcular un promedio.

**Asignar cero al promedio introduciría información artificial**, pues podría interpretarse como una valoración negativa cuando, en realidad, la película no ha recibido calificaciones.

Este criterio debe mantenerse en las demás métricas: cuando no existan observaciones que permitan calcular un indicador, su ausencia debe representarse de acuerdo con su significado estadístico y no mediante valores arbitrarios.

## Beneficios de la estrategia de agregación

La construcción previa de métricas a nivel de película permite:

- Preservar la granularidad original del catálogo.
- Evitar uniones many-to-many accidentales.
- Reducir el volumen de datos durante las integraciones.
- Facilitar los análisis descriptivos y las visualizaciones posteriores.
- Mantener la trazabilidad de las transformaciones.
- Disponer de indicadores listos para su utilización en análisis posteriores.

## Conclusiones

La agregación constituye una etapa intermedia entre los datos transaccionales y la construcción de estructuras analíticas.

Como resultado:

- Se resume la actividad de las tablas `ratings` y `tags` a nivel de película.
- Se generan indicadores descriptivos de calificaciones y etiquetas.
- Se verifica la unicidad de `movieId` en ambas tablas agregadas.
- Se preserva la granularidad del catálogo y se evitan duplicaciones durante las integraciones.
- Se establece un criterio adecuado para representar métricas sin observaciones.

Las tablas `rating_agg` y `tag_agg` servirán como componentes del catálogo enriquecido y de los conjuntos de datos analíticos de las siguientes etapas del proyecto.

## Construcción de la dimensión consolidada de películas

Tras completar la validación de calidad de los datos, normalizar la información de géneros y generar métricas agregadas a partir de las tablas transaccionales, se procede a construir una dimensión consolidada de películas.

El objetivo de esta etapa consiste en integrar en una única estructura toda la información relevante asociada a cada película, preservando en todo momento la granularidad original del catálogo.

La tabla resultante, denominada `movies_consolidated`, se convertirá en uno de los principales activos analíticos del proyecto, ya que permitirá estudiar características descriptivas, actividad de usuarios y métricas derivadas sin necesidad de realizar uniones complejas sobre millones de registros transaccionales.

## Objetivo de la integración

Las tablas analizadas hasta este punto contienen distintos tipos de información:

| Tabla | Información principal |
|---------|---------|
| `movies` | Catálogo de películas |
| `links` | Identificadores externos |
| `rating_agg` | Métricas agregadas de calificaciones |
| `tag_agg` | Métricas agregadas de etiquetas |

Cada una aporta una perspectiva diferente sobre una misma entidad: la película.

La integración busca reunir esta información en una estructura común donde cada observación continúe representando exactamente una película.

## Definición del universo de referencia

Un aspecto crítico de esta etapa consiste en determinar cuál de las tablas actuará como universo de referencia durante la integración.

En este proyecto se adopta como tabla principal:

```text
movies
```

porque representa el catálogo oficial distribuido por MovieLens.

Esto implica que todas las películas presentes en el catálogo deben conservarse independientemente de si poseen actividad registrada o no.

### Justificación

Existen películas que:

- No han recibido calificaciones.
- No han sido etiquetadas.
- Carecen parcialmente de identificadores externos.

Sin embargo, dichas películas siguen formando parte legítima del catálogo.

Por esta razón, eliminar registros únicamente por ausencia de actividad generaría una representación incompleta del universo de películas disponible.

## Estrategia de integración

La construcción de `movies_consolidated` se realiza utilizando un conjunto de uniones tipo:

```text
LEFT JOIN
```

tomando como punto de partida la tabla `movies`.

Este enfoque garantiza que:

- Todas las películas del catálogo permanezcan en la tabla final.
- Ninguna película sea eliminada por falta de actividad.
- Las métricas agregadas se incorporen únicamente cuando exista información disponible.
- La granularidad original se conserve durante todo el proceso.

## Validación explícita de relaciones

Cada integración utiliza el parámetro:

```text
validate="one_to_one"
```

que obliga a Pandas a comprobar que ambas tablas cumplen una relación uno a uno sobre la llave utilizada.

Esta validación constituye una protección adicional contra errores metodológicos frecuentes, tales como:

- Uniones many-to-many accidentales.
- Duplicación de registros.
- Multiplicación no deseada del catálogo.
- Alteración inadvertida de la granularidad de análisis.

Si alguna de las tablas incumpliera esta condición, el proceso se interrumpiría automáticamente.

## Tratamiento de métricas derivadas

Una vez completada la integración, algunas películas presentan valores faltantes en las métricas agregadas debido a que no registran actividad de usuarios.

Es importante distinguir dos tipos de variables:

### Métricas de conteo

Las variables:

- `rating_count`
- `rating_users`
- `tag_application_count`
- `non_null_tag_count`
- `distinct_tags`
- `tag_users`

representan recuentos observables.

Cuando una película no registra actividad, el valor correcto es:

```text
0
```

Por esta razón, los valores faltantes asociados a estas variables se sustituyen explícitamente por cero.

### Métricas estadísticas

Indicadores como:

- `rating_mean`
- `rating_std`
- `rating_min`
- `rating_max`

requieren la existencia de observaciones para ser calculados.

En ausencia de calificaciones, estos valores permanecen como:

```text
NaN
```

ya que no existe evidencia suficiente para estimarlos.

### Principio metodológico

La ausencia de calificaciones no implica una valoración de cero.

Por ejemplo:

```text
rating_count = 0
```

no significa que:

```text
rating_mean = 0
```

sino que el promedio es desconocido debido a la falta de observaciones.

Conservar los nulos evita introducir información artificial en el proceso analítico.

## Construcción de variables indicadoras

Además de las métricas agregadas, se generan indicadores booleanos que facilitan análisis posteriores.

### Disponibilidad de calificaciones

```text
has_ratings
```

Indica si la película posee al menos una calificación registrada.

### Disponibilidad de etiquetas

```text
has_tags
```

Indica si la película ha recibido alguna etiqueta.

### Disponibilidad de identificadores externos

```text
has_imdb_id
```

Permite identificar películas con correspondencia en IMDb.

```text
has_tmdb_id
```

Permite identificar películas con correspondencia en TMDb.

Estas variables simplifican futuros análisis de cobertura y permiten construir filtros sin recalcular condiciones complejas.

## Verificación de la integridad de la integración

Tras finalizar el proceso se realiza una auditoría para asegurar que la estructura del catálogo no haya sido alterada.

Se evalúan cuatro condiciones fundamentales.

### Conservación del número de filas

La cantidad de registros en la tabla consolidada debe coincidir exactamente con el número de películas presentes en el catálogo original.

Resultado observado:

```text
movies_original_rows = 87,585
consolidated_rows = 87,585
```

### Conservación de unicidad

El identificador principal debe seguir siendo único.

Resultado:

```text
consolidated_movieId_unique = True
```

### Preservación completa del catálogo

No debe perderse ninguna película durante la integración.

Resultado:

```text
rows_preserved = True
```

### Ausencia de alteraciones de granularidad

La combinación de validaciones anteriores confirma que la tabla consolidada mantiene exactamente una fila por película.

Esto garantiza que las futuras operaciones de análisis podrán realizarse sobre una dimensión estable y consistente.

In [ ]:
movies_base = movies.drop(columns=["title_norm"], errors="ignore").copy()
movies_base["year"]=(movies_base["title"]
                   .str.extract(r"\((\d{4})\)\s*$")[0]
                   .astype("Int64"))
movies_base["title_clean"] = (movies_base["title"]
                            .str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
                            .str.strip())
movies_base= movies_base[["movieId", "title_clean", "year", "genres"]]

movies_consolidated = (
    movies_base
    .merge(links, on="movieId", how="left", validate="one_to_one")
    .merge(rating_agg, on="movieId", how="left", validate="one_to_one")
    .merge(tag_agg, on="movieId", how="left", validate="one_to_one")
)

count_columns = [
    "rating_count", "rating_users",
    "tag_application_count", "non_null_tag_count",
    "distinct_tags", "tag_users"
]
for col in count_columns:
    movies_consolidated[col] = movies_consolidated[col].fillna(0).astype("int64")

movies_consolidated["has_ratings"] = movies_consolidated["rating_count"].gt(0)
movies_consolidated["has_tags"] = movies_consolidated["tag_application_count"].gt(0)
movies_consolidated["has_imdb_id"] = movies_consolidated["imdbId"].notna()
movies_consolidated["has_tmdb_id"] = movies_consolidated["tmdbId"].notna()

integration_check = {
    "movies_original_rows": len(movies),
    "consolidated_rows": len(movies_consolidated),
    "consolidated_movieId_unique": bool(movies_consolidated["movieId"].is_unique),
    "rows_preserved": len(movies) == len(movies_consolidated)
}
display(pd.Series(integration_check, name="result"))

if not integration_check["rows_preserved"] or not integration_check["consolidated_movieId_unique"]:
    raise AssertionError("La integración alteró la granularidad del catálogo.")

movies_original_rows           87585
consolidated_rows              87585
consolidated_movieId_unique     True
rows_preserved                  True
Name: result, dtype: object

In [ ]:
movies_consolidated.head()

,movieId,title_clean,year,genres,imdbId,tmdbId,rating_count,rating_mean,rating_std,rating_min,rating_max,rating_users,tag_application_count,non_null_tag_count,distinct_tags,tag_users,has_ratings,has_tags,has_imdb_id,has_tmdb_id
0,1,Toy Story,1995,Adventure|Animation|Children|Comedy|Fantasy,114709,862,68997,3.8974,0.9215,0.5000,5.0000,68997,1230,1230,435,255,True,True,True,True
1,2,Jumanji,1995,Adventure|Children|Fantasy,113497,8844,28904,3.2758,0.9555,0.5000,5.0000,28904,573,573,348,83,True,True,True,True
2,3,Grumpier Old Men,1995,Comedy|Romance,113228,15602,13134,3.1394,1.0123,0.5000,5.0000,13134,23,23,23,15,True,True,True,True
3,4,Waiting to Exhale,1995,Comedy|Drama|Romance,114885,31357,2806,2.8453,1.1059,0.5000,5.0000,2806,12,12,9,6,True,True,True,True
4,5,Father of the Bride Part II,1995,Comedy,113041,11862,13154,3.0596,0.9991,0.5000,5.0000,13154,64,64,35,14,True,True,True,True


## Resultado obtenido

La tabla `movies_consolidated` constituye ahora una vista enriquecida del catálogo MovieLens que integra:

- Información descriptiva de las películas.
- Identificadores externos.
- Métricas de calificaciones.
- Métricas de etiquetado.
- Indicadores de disponibilidad de actividad.

Todo ello preservando la estructura:

```text
1 película = 1 fila
```

que constituye el principio fundamental de esta dimensión.

## Conclusiones

La construcción de `movies_consolidated` representa uno de los hitos principales de la fase de integración de datos.

Los resultados obtenidos demuestran que:

- La integración preservó completamente el catálogo original.
- No se introdujeron duplicaciones de películas.
- Se mantuvo la unicidad de `movieId`.
- Las métricas agregadas se incorporaron de forma segura.
- Los conteos se diferenciaron correctamente de las métricas estadísticas.
- La información quedó preparada para análisis exploratorios y comparativos posteriores.

Con esta nueva estructura se dispone de una dimensión analítica robusta que permitirá estudiar el comportamiento de las películas desde múltiples perspectivas sin depender directamente de los millones de registros presentes en las tablas transaccionales originales.

## Construcción de tablas de eventos enriquecidas y estrategia de almacenamiento analítico

Hasta este punto del proceso se han construido dos tipos de estructuras complementarias:

- Una **dimensión consolidada de películas** (`movies_consolidated`), donde existe exactamente una fila por película.
- Un conjunto de **tablas transaccionales** (`ratings` y `tags`) que conservan el detalle completo de las interacciones realizadas por los usuarios.

Aunque la dimensión consolidada resulta ideal para análisis agregados a nivel de película, existen numerosos escenarios analíticos donde es necesario conservar el nivel de detalle original de cada evento.

Por ejemplo:

- Analizar la evolución temporal de las calificaciones.
- Estudiar el comportamiento individual de los usuarios.
- Examinar patrones de etiquetado.
- Construir modelos de recomendación.
- Analizar secuencias de interacción entre usuarios y películas.

Para estos casos resulta conveniente disponer de tablas enriquecidas que combinen los eventos originales con la información descriptiva de cada película.

## Objetivo de las tablas enriquecidas

El propósito de esta etapa consiste en incorporar información contextual a los eventos registrados en las tablas transaccionales.

Se generan dos nuevas estructuras:

### `ratings_enriched`

Conserva la granularidad original de una fila por calificación.

Cada registro mantiene:

- Identificador de usuario.
- Identificador de película.
- Calificación otorgada.
- Marca temporal original.

y añade:

- Título de la película.
- Géneros asociados.

### `tags_enriched`

Conserva la granularidad original de una fila por aplicación de etiqueta.

Cada registro mantiene:

- Identificador de usuario.
- Identificador de película.
- Texto de la etiqueta.
- Marca temporal original.

y añade:

- Título de la película.
- Géneros asociados.

De esta forma, los análisis posteriores pueden realizarse sin necesidad de consultar continuamente la dimensión de películas.

## Principio de separación entre eventos

Un aspecto metodológico importante consiste en mantener separadas las tablas de ratings y tags.

Aunque ambas describen actividad de usuarios sobre películas, representan fenómenos diferentes y poseen granularidades propias.

### Riesgo de una integración directa

Unir directamente las tablas `ratings` y `tags` utilizando únicamente `movieId` produciría una expansión artificial del número de registros.

Por ejemplo:

- Una película con 1,000 calificaciones.
- Una película con 50 etiquetas.

Generaría potencialmente:

```text
1,000 × 50 = 50,000 filas
```

tras una unión many-to-many.

Este fenómeno no representa información nueva y únicamente introduce duplicaciones derivadas del proceso de integración.

Por esta razón:

> Las tablas `ratings_enriched` y `tags_enriched` se construyen y almacenan por separado.

## Construcción de una dimensión mínima de consulta

Para enriquecer las tablas transaccionales no se utiliza la tabla consolidada completa.

En su lugar se genera una dimensión mínima que contiene únicamente:

- `movieId`
- `title`
- `genres`

Esta estrategia ofrece varias ventajas:

- Reduce el consumo de memoria.
- Simplifica las operaciones de integración.
- Evita incorporar columnas innecesarias.
- Mejora el rendimiento durante el procesamiento de grandes volúmenes de datos.

La dimensión funciona como una tabla de consulta ligera utilizada exclusivamente para enriquecer los eventos.

## Procesamiento por bloques

Uno de los principales desafíos del conjunto MovieLens 32M es el tamaño de la tabla de calificaciones.

Con más de treinta millones de registros, realizar una integración completa en memoria podría:

- Incrementar significativamente el consumo de RAM.
- Reducir el rendimiento.
- Limitar la reproducibilidad en equipos con recursos moderados.

Por esta razón se adopta una estrategia de procesamiento incremental basada en bloques (`chunks`).

### Funcionamiento del procesamiento por bloques

Los datos se leen en fragmentos de tamaño controlado.

Cada bloque:

1. Se carga desde el archivo original.
2. Se integra con la dimensión de películas.
3. Se transforma.
4. Se escribe inmediatamente al almacenamiento de salida.

Posteriormente el bloque es liberado de memoria y el proceso continúa con el siguiente fragmento.

Este enfoque permite procesar conjuntos de datos muy grandes sin necesidad de mantener una copia completa del dataset enriquecido en memoria.

## Incorporación de variables temporales

La documentación original distribuye las marcas de tiempo en formato Unix.

Por motivos de trazabilidad se conserva la columna:

```text
timestamp
```

exactamente como aparece en los archivos fuente.

Sin embargo, para facilitar futuros análisis temporales, se añade una segunda representación:

```text
timestamp_utc
```

obtenida mediante la conversión del timestamp Unix a un objeto datetime en UTC.

Esta estrategia ofrece dos ventajas importantes:

### Conservación del dato original

Se mantiene intacta la información distribuida por MovieLens.

### Facilidad de análisis

Los procesos posteriores pueden trabajar directamente con fechas interpretables sin necesidad de repetir conversiones.

## Estrategia de almacenamiento

Los conjuntos enriquecidos pueden alcanzar tamaños considerablemente mayores que las tablas originales debido a la incorporación de información descriptiva adicional.

Por esta razón, el sistema prioriza el uso de formatos analíticos más eficientes.

### Uso preferente de Parquet

Cuando existe un motor compatible disponible (`pyarrow` o `fastparquet`), los datos se almacenan utilizando formato:

```text
Parquet
```

Parquet proporciona diversas ventajas para análisis de datos:

- Menor tamaño de almacenamiento.
- Lectura más rápida.
- Compresión eficiente.
- Optimización para procesamiento columnar.
- Integración nativa con Pandas y herramientas de ciencia de datos.

Además, cada bloque procesado puede almacenarse como un fragmento independiente dentro de un dataset Parquet, evitando la necesidad de generar un archivo monolítico de gran tamaño.

### Alternativa CSV

Si no existe un motor Parquet disponible en el entorno de ejecución, el procedimiento utiliza CSV como mecanismo alternativo.

En este caso:

- El archivo se construye progresivamente.
- El encabezado se escribe una única vez.
- Los bloques posteriores se anexan secuencialmente.

Aunque menos eficiente que Parquet, esta estrategia garantiza la portabilidad y reproducibilidad del proyecto.

In [ ]:
# Dimensión mínima de película para enriquecer tablas de eventos.
movie_lookup = movies[["movieId", "title", "genres"]].copy()

def write_enriched_in_chunks(input_path, output_stem, table_kind, chunksize=500_000):
    """Enriquece una tabla de eventos por bloques sin cargar otra copia completa."""
    output_parquet = PROCESSED_DIR / f"{output_stem}.parquet"
    output_csv = PROCESSED_DIR / f"{output_stem}.csv"

    # Preferir Parquet si existe un motor compatible.
    try:
        import pyarrow  # noqa: F401
        parquet_available = True
    except ImportError:
        try:
            import fastparquet  # noqa: F401
            parquet_available = True
        except ImportError:
            parquet_available = False

    if parquet_available:
        # Escribir fragmentos Parquet independientes y después combinarlos como dataset.
        # Esta estrategia evita requerir un único archivo Parquet monolítico.
        fragment_dir = PROCESSED_DIR / f"{output_stem}_parquet"
        fragment_dir.mkdir(parents=True, exist_ok=True)
        for old_file in fragment_dir.glob("part-*.parquet"):
            old_file.unlink()

        for i, chunk in enumerate(pd.read_csv(input_path, chunksize=chunksize)):
            chunk = chunk.merge(movie_lookup, on="movieId", how="left", validate="many_to_one")
            chunk["timestamp_utc"] = pd.to_datetime(
                chunk["timestamp"], unit="s", utc=True, errors="coerce"
            )
            chunk.to_parquet(
                fragment_dir / f"part-{i:05d}.parquet",
                index=False
            )
        return fragment_dir

    # Alternativa CSV: escribir encabezado una sola vez.
    if output_csv.exists():
        output_csv.unlink()

    for i, chunk in enumerate(pd.read_csv(input_path, chunksize=chunksize)):
        chunk = chunk.merge(movie_lookup, on="movieId", how="left", validate="many_to_one")
        chunk["timestamp_utc"] = pd.to_datetime(
            chunk["timestamp"], unit="s", utc=True, errors="coerce"
        )
        chunk.to_csv(
            output_csv,
            mode="w" if i == 0 else "a",
            header=(i == 0),
            index=False,
            encoding="utf-8"
        )
    return output_csv

ratings_enriched_path = write_enriched_in_chunks(
    FILES["ratings"], "ratings_enriched", "ratings"
)
tags_enriched_path = write_enriched_in_chunks(
    FILES["tags"], "tags_enriched", "tags"
)

print("Ratings enriquecidos:", ratings_enriched_path)
print("Tags enriquecidos:", tags_enriched_path)

Ratings enriquecidos: c:\Users\igpea\OneDrive\Documents\Diplomado\Proyecto_Integrador_M1\data\processed\ratings_enriched.csv
Tags enriquecidos: c:\Users\igpea\OneDrive\Documents\Diplomado\Proyecto_Integrador_M1\data\processed\tags_enriched.csv


## Resultados obtenidos

La ejecución del procedimiento genera satisfactoriamente: `ratings_enriched` y `tags_enriched`

dentro del directorio: `data/processed/`

En el entorno actual, ambos conjuntos fueron almacenados como archivos CSV:

```text
ratings_enriched.csv
tags_enriched.csv
```

quedando disponibles para análisis posteriores.

## Beneficios de la estrategia implementada

La construcción de tablas de eventos enriquecidas aporta múltiples ventajas analíticas y operativas:

- Preserva la granularidad original de los eventos.
- Evita uniones many-to-many accidentales.
- Facilita consultas descriptivas posteriores.
- Incorpora contexto de película directamente en los registros.
- Conserva las marcas temporales originales.
- Añade representaciones temporales listas para análisis.
- Permite trabajar con volúmenes masivos de datos mediante procesamiento por bloques.
- Optimiza el almacenamiento mediante formatos analíticos cuando están disponibles.

## Conclusiones

Con esta etapa se completa la construcción de los principales activos analíticos derivados del conjunto MovieLens.

A partir de este momento el proyecto dispone de:

- Una dimensión consolidada de películas (`movies_consolidated`).
- Una tabla puente normalizada película–género (`movie_genres`).
- Una tabla enriquecida de calificaciones (`ratings_enriched`).
- Una tabla enriquecida de etiquetas (`tags_enriched`).

Estas estructuras conservan la integridad del modelo original, preservan las distintas granularidades de análisis y proporcionan una base sólida para las etapas posteriores de exploración visual, análisis estadístico y construcción de modelos analíticos.

## Validaciones posteriores a la limpieza e integración

Una vez completadas las etapas de limpieza, normalización e integración de datos, resulta imprescindible verificar que las transformaciones realizadas no hayan alterado involuntariamente la estructura original del modelo de datos.

Esta fase funciona como una etapa formal de aseguramiento de calidad y tiene como propósito comprobar que los productos generados conservan las propiedades esperadas de unicidad, integridad referencial y consistencia numérica.

A diferencia de las validaciones exploratorias realizadas anteriormente, estas comprobaciones actúan como controles de aceptación del proceso. Si alguna condición crítica no se cumple, la ejecución del notebook se interrumpe automáticamente para evitar que errores estructurales se propaguen hacia etapas posteriores de análisis.

## Objetivo de las validaciones finales

Durante las fases de limpieza e integración se generaron nuevas estructuras derivadas:

- La dimensión consolidada de películas (`movies_consolidated`).
- La tabla puente película–género (`movie_genres`).
- Los agregados de calificaciones (`rating_agg`).
- Los agregados de etiquetas (`tag_agg`).

Cada una de estas estructuras debe cumplir ciertas reglas de negocio y consistencia derivadas del diseño original del dataset.

Las validaciones finales tienen como objetivo comprobar que:

- No se perdieron registros durante la integración.
- No se introdujeron duplicados.
- Las relaciones siguen siendo válidas.
- Los agregados representan exactamente los datos originales.
- Se preservó la granularidad prevista para cada tabla.

## Verificación de la dimensión consolidada de películas

La tabla `movies_consolidated` constituye el principal producto de integración generado durante el proceso.

Como se definió previamente, esta estructura debe mantener una relación:

```text
1 película = 1 fila
```

Para comprobarlo se evalúan dos condiciones fundamentales.

### Unicidad de `movieId`

El identificador de película debe seguir siendo único después de integrar información procedente de múltiples fuentes.

Esta validación permite detectar:

- Duplicaciones introducidas por errores de integración.
- Relaciones many-to-many accidentales.
- Expansiones no deseadas del catálogo.

Resultado:

```text
movieId único en movies_consolidated → Correcto
```

### Conservación del número de películas

La cantidad total de registros en la tabla consolidada debe coincidir exactamente con la cantidad de películas presentes en el catálogo original.

Esto debe confirmar que ninguna película fue eliminada ni duplicada durante el proceso de integración.

## Validación de la tabla puente película–género

La tabla `movie_genres` constituye una representación normalizada de la relación entre películas y géneros.

Debido a su naturaleza relacional, resulta necesario verificar dos propiedades esenciales.

### Unicidad de la combinación `(movieId, genre)`

Cada combinación película–género debe aparecer una sola vez.

La existencia de duplicados en esta relación produciría:

- Sobreestimación de frecuencias.
- Distorsiones en agregaciones posteriores.
- Errores en análisis de distribución por género.

Resultado:

```text
Combinación movieId–genre única → Correcto
```

### Integridad referencial de géneros

Toda película presente en la tabla puente debe existir previamente dentro del catálogo principal.

Para comprobarlo se verifica que todos los `movieId` registrados en `movie_genres` pertenezcan al conjunto de películas válidas del catálogo.

Resultado:

```text
Todos los géneros están vinculados a películas existentes
```

No se detectan identificadores huérfanos.

## Validación de los agregados de actividad

Las tablas agregadas representan resúmenes construidos a partir de millones de registros transaccionales.

Por esta razón resulta fundamental confirmar que:

- Mantienen una fila por película.
- Conservan correctamente la información resumida.
- No se han perdido observaciones durante el proceso de agrupación.

### Unicidad de los agregados

Las tablas:

- `rating_agg`
- `tag_agg`

deben contener una única observación por `movieId`.

Esta propiedad garantiza que puedan integrarse posteriormente mediante relaciones uno a uno.

La validación confirma que ambas tablas cumplen esta condición.

## Reconciliación de conteos

Una de las verificaciones más importantes consiste en comprobar que las métricas agregadas reconstruyan exactamente el volumen original de eventos.

### Reconciliación de calificaciones

La suma de:

```text
rating_count
```

debe ser exactamente igual al número total de filas presentes en la tabla `ratings`.

Conceptualmente:

```text
Σ rating_count = total de calificaciones originales
```

Resultado:

```text
Agregado de ratings reconcilia con ratings original → Correcto
```

Esto demuestra que ninguna calificación fue omitida o duplicada durante el procedimiento de agregación.

### Reconciliación de etiquetas

De forma análoga, la suma de:

```text
tag_application_count
```

debe coincidir con el número total de registros de la tabla `tags`.

Resultado:

```text
Agregado de tags reconcilia con tags original → Correcto
```

Por tanto, todas las aplicaciones de etiquetas fueron incorporadas correctamente a la tabla agregada.

## Estrategia de control mediante afirmaciones (`assert`)

Las verificaciones críticas se implementan utilizando instrucciones `assert`.

Este mecanismo permite transformar las reglas metodológicas del proyecto en condiciones obligatorias de ejecución.

Si cualquiera de estas condiciones falla, el notebook detiene automáticamente su ejecución y genera una excepción.

Entre las ventajas de este enfoque se encuentran:

- Detección temprana de errores.
- Prevención de resultados inconsistentes.
- Automatización de controles de calidad.
- Mayor reproducibilidad del proceso.

De este modo, la validez de los datos no depende únicamente de inspecciones visuales sino también de comprobaciones programáticas verificables.

In [ ]:
assert movies_consolidated["movieId"].is_unique
assert len(movies_consolidated) == len(movies)
assert not movie_genres.duplicated(["movieId", "genre"]).any()
assert set(movie_genres["movieId"]).issubset(catalog_movie_ids)
assert rating_agg["movieId"].is_unique
assert tag_agg["movieId"].is_unique

# Reconciliación de conteos.
assert int(rating_agg["rating_count"].sum()) == len(ratings)
assert int(tag_agg["tag_application_count"].sum()) == len(tags)

post_validation = pd.DataFrame([
    {"test": "movieId único en movies_consolidated",
     "passed": movies_consolidated["movieId"].is_unique},
    {"test": "Se preservó el total de películas",
     "passed": len(movies_consolidated) == len(movies)},
    {"test": "Combinación movieId-genre única",
     "passed": not movie_genres.duplicated(["movieId", "genre"]).any()},
    {"test": "Géneros vinculados a películas existentes",
     "passed": set(movie_genres["movieId"]).issubset(catalog_movie_ids)},
    {"test": "Agregado de ratings reconcilia con ratings original",
     "passed": int(rating_agg["rating_count"].sum()) == len(ratings)},
    {"test": "Agregado de tags reconcilia con tags original",
     "passed": int(tag_agg["tag_application_count"].sum()) == len(tags)}
])
display(post_validation)

if not post_validation["passed"].all():
    raise AssertionError("Una o más validaciones posteriores fallaron.")

,test,passed
0,movieId único en movies_consolidated,True
1,Se preservó el total de películas,True
2,Combinación movieId-genre única,True
3,Géneros vinculados a películas existentes,True
4,Agregado de ratings reconcilia con ratings ori...,True
5,Agregado de tags reconcilia con tags original,True


## Resultados obtenidos

Las seis validaciones ejecutadas fueron superadas satisfactoriamente. No se identificó ninguna inconsistencia estructural en los datos procesados.

## Conclusiones

Las validaciones posteriores confirman que el proceso de limpieza e integración preservó correctamente las propiedades fundamentales del modelo de datos.

Los principales hallazgos son:

- La granularidad de la dimensión de películas permanece intacta.
- No se introdujeron duplicaciones durante las integraciones.
- La tabla puente mantiene integridad referencial completa.
- Los agregados reproducen exactamente el volumen de eventos originales.
- No se detectaron pérdidas de información durante las transformaciones.

En consecuencia, las estructuras generadas pueden considerarse productos analíticos válidos y confiables para las siguientes etapas del proyecto, incluyendo exploración visual, análisis descriptivo avanzado y generación de modelos analíticos.

## Exportación de resultados y generación de artefactos reproducibles

Una vez completadas las etapas de validación, normalización, agregación e integración de datos, se procede a materializar los productos generados mediante la exportación de los resultados a la capa de datos procesados del proyecto.

Esta etapa marca la transición entre el proceso de preparación de datos y las futuras fases de exploración, visualización y análisis estadístico.

El objetivo principal es almacenar de forma estructurada todos los artefactos analíticos generados, permitiendo que puedan ser reutilizados posteriormente sin necesidad de repetir cada transformación intermedia.

## Principios de almacenamiento adoptados

La estrategia de exportación sigue tres principios fundamentales:

### Reproducibilidad

Todos los archivos exportados son productos derivados que pueden reconstruirse ejecutando nuevamente el notebook.

No se generan transformaciones manuales ni intervenciones externas sobre los archivos resultantes.

### Trazabilidad

Cada archivo exportado responde a una finalidad analítica específica y conserva una relación directa con alguna de las etapas desarrolladas durante el proceso de limpieza e integración.

Esto facilita auditar posteriormente cómo se generó cada resultado.

### Separación entre datos originales y datos procesados

Los archivos originales distribuidos por MovieLens permanecen intactos dentro de:

```text
data/raw/
```

mientras que todos los productos generados por el proyecto se almacenan en:

```text
data/processed/
```

Esta separación evita modificaciones accidentales sobre las fuentes originales de información y permite reconstruir completamente cualquier etapa del procesamiento.

## Exportación de estructuras analíticas principales

### Catálogo consolidado de películas

```text
movies_consolidated.csv
```

Constituye el principal producto analítico generado durante esta fase.

Posee exactamente una fila por película e integra:

- Información descriptiva.
- Identificadores externos.
- Métricas agregadas de calificaciones.
- Métricas agregadas de etiquetas.
- Indicadores de disponibilidad de actividad.

Su propósito es servir como dimensión enriquecida para análisis posteriores a nivel de película.

---

### Tabla puente película–género

```text
movie_genres.csv
```

Contiene una fila por combinación:

```text
movieId – genre
```

Esta estructura normalizada permite realizar análisis por género sin necesidad de procesar nuevamente la columna multivaluada `genres`.

Entre sus posibles aplicaciones se encuentran:

- Distribuciones por género.
- Comparaciones entre categorías cinematográficas.
- Métricas agregadas segmentadas por género.

---

### Métricas agregadas de calificaciones

```text
rating_aggregates_by_movie.csv
```

Resume la actividad de calificación observada para cada película.

Incluye indicadores como:

- Número de calificaciones.
- Promedio.
- Desviación estándar.
- Valores mínimos y máximos.
- Número de usuarios calificadores.

Esta tabla permite realizar análisis descriptivos sin necesidad de consultar los millones de registros contenidos en `ratings`.

---

### Métricas agregadas de etiquetas

```text
tag_aggregates_by_movie.csv
```

Contiene indicadores derivados de la actividad de etiquetado.

Entre otras métricas se incluyen:

- Número total de aplicaciones de etiquetas.
- Número de etiquetas distintas.
- Número de usuarios etiquetadores.

Esta información complementa los indicadores de interacción de usuarios sobre las películas.

## Exportación de diagnósticos de cobertura

### Películas sin calificaciones

```text
movies_without_ratings.csv
```

Incluye las películas pertenecientes al catálogo que no presentan actividad de calificación.

El archivo puede utilizarse para:

- Evaluar cobertura del catálogo.
- Analizar sesgos de participación.
- Identificar películas con baja interacción.

---

### Películas sin etiquetas

```text
movies_without_tags.csv
```

Contiene las películas que no poseen registros de etiquetado.

Permite estudiar diferencias entre películas frecuentemente descritas por los usuarios y aquellas que permanecen sin anotaciones semánticas.

## Exportación de diagnósticos de calidad

### Resumen general de calidad

```text
quality_overview.csv
```

Almacena los indicadores globales calculados para cada tabla:

- Número de registros.
- Número de columnas.
- Valores nulos.
- Duplicados.
- Completitud.
- Consumo de memoria.

Constituye una referencia rápida del estado general de los datos.

---

### Perfil de calidad por columna

```text
column_quality.csv
```

Contiene métricas individuales para cada variable:

- Tipo de dato.
- Valores nulos.
- Porcentaje de completitud.
- Cardinalidad.
- Conteos de observaciones válidas.

Este archivo facilita futuras auditorías de calidad.

---

### Integridad referencial

```text
referential_integrity.csv
```

Documenta las verificaciones realizadas sobre las correspondencias de `movieId` entre las distintas tablas del modelo.

Permite demostrar la consistencia relacional observada durante el proceso de integración.

## Exportación de diagnósticos complementarios

Además de los productos principales, se almacenan diversos resultados generados durante la fase de validación.

Entre ellos destacan:

### Validación de llaves

```text
key_checks.csv
```

Resume las comprobaciones realizadas sobre identificadores y llaves candidatas.

---

### Diagnóstico de títulos repetidos

```text
title_duplicates_exact.csv
```

Documenta títulos repetidos detectados durante la exploración del catálogo.

---

### Distribución de géneros

```text
genre_frequency.csv
```

Contiene la frecuencia observada de cada género cinematográfico dentro del catálogo.

---

### Distribución de calificaciones

```text
rating_distribution.csv
```

Registra las frecuencias y proporciones de cada valor de la escala de calificación.

---

### Resultados de validación final

```text
post_validation.csv
```

Almacena las comprobaciones de aceptación realizadas al concluir el proceso de integración.

## Exportación de diagnósticos especializados

Algunos resultados se almacenan dentro del directorio:

```text
data/processed/diagnostics/
```

con el objetivo de separar los productos analíticos de los artefactos de control de calidad.

### Reporte de valores nulos

```text
null_report.csv
```

Contiene el detalle de las variables que presentan valores faltantes.

---

### Películas sin identificador TMDb

```text
links_without_tmdb.csv
```

Documenta los registros donde la correspondencia con TMDb no se encuentra disponible.

---

### Etiquetas sin contenido textual

```text
tags_without_text.csv
```

Almacena los registros de etiquetado donde el atributo `tag` presenta valores nulos.

La conservación de estos registros facilita futuras auditorías sin alterar las tablas base.

## Tablas enriquecidas de eventos

Las estructuras:

```text
ratings_enriched
```

y

```text
tags_enriched
```

también forman parte de los productos generados durante esta fase.

Dependiendo de las bibliotecas disponibles en el entorno, estas tablas pueden almacenarse mediante dos estrategias:

### Formato Parquet

Cuando existe soporte para:

- `pyarrow`
- `fastparquet`

los datos se exportan como datasets Parquet particionados.

Este formato ofrece importantes ventajas:

- Menor tamaño de almacenamiento.
- Mayor velocidad de lectura.
- Mejor rendimiento analítico.

### Formato CSV

Cuando no existe un motor Parquet disponible, los datos se exportan como archivos CSV.

Aunque menos eficiente, esta alternativa garantiza portabilidad y compatibilidad universal.

## Resultado final de la etapa

La ejecución satisfactoria de esta fase confirma que todos los productos derivados fueron almacenados correctamente dentro del directorio:

```text
data/processed/
```

A partir de este momento el proyecto dispone de un conjunto completo de:

- Tablas consolidadas.
- Tablas normalizadas.
- Métricas agregadas.
- Diagnósticos de calidad.
- Validaciones estructurales.
- Eventos enriquecidos.

Todos ellos reproducibles, documentados y listos para ser utilizados durante las siguientes etapas de exploración visual, análisis estadístico y generación de conocimiento sobre el comportamiento de usuarios y películas dentro del ecosistema MovieLens.

In [ ]:
# Exportar tablas principales.
exports = {
    "movies_consolidated.csv": movies_consolidated,
    "movie_genres.csv": movie_genres_imp,
    "rating_aggregates_by_movie.csv": rating_agg,
    "tag_aggregates_by_movie.csv": tag_agg,
    "movies_without_ratings.csv": movies_without_ratings,
    "movies_without_tags.csv": movies_without_tags,
    "quality_overview.csv": overview,
    "column_quality.csv": column_quality,
    "referential_integrity.csv": referential_report,
    "key_checks.csv": key_checks,
    "title_duplicates_exact.csv": title_counts,
    "genre_frequency.csv": genre_frequency,
    "rating_distribution.csv": rating_distribution,
    "post_validation.csv": post_validation
}

for filename, df in exports.items():
    df.to_csv(PROCESSED_DIR / filename, index=False, encoding="utf-8")

null_report.to_csv(
    DIAGNOSTICS_DIR / "null_report.csv", index=False, encoding="utf-8"
)
tmdb_missing.to_csv(
    DIAGNOSTICS_DIR / "links_without_tmdb.csv", index=False, encoding="utf-8"
)
tags_null.to_csv(
    DIAGNOSTICS_DIR / "tags_without_text.csv", index=False, encoding="utf-8"
)

print(f"Exportación completada en: {PROCESSED_DIR}")

Exportación completada en: c:\Users\igpea\OneDrive\Documents\Diplomado\Proyecto_Integrador_M1\data\processed
